In [9]:
!pip install -q wandb polars xgboost imblearn optuna networkit geonamescache pycountry

In [10]:
# pytorch 라이브러리 설치
!pip install -q torch_geometric

In [11]:
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv

In [41]:
import wandb
wandb.login()
# 계정 만들었으면, 2 누르고 본인 api 키 입력

True

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1) Imports


In [14]:
import os, gc, time, copy, math, random
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import MessagePassing

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)
from tqdm.auto import tqdm
import optuna

import re
import pycountry
import geonamescache

from sklearn import set_config
from collections import deque
from datetime import timedelta


import torch
from torch import Tensor
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter, Linear
from typing import Union, Tuple, Optional

from torch_geometric.typing import (Adj, Size, OptTensor, PairTensor)
from torch_sparse import SparseTensor, set_diag
from torch_geometric.data import Data
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.utils import remove_self_loops, add_self_loops, softmax
from torch_geometric.nn.inits import glorot, zeros
from torch_geometric.loader import NeighborLoader

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


# 2) 데이터 로드 + Timestamp + Split


In [15]:
master_parquet = "/content/drive/MyDrive/data/parquet/HI-Medium_Master_v2.parquet"
q_master = pl.scan_parquet(master_parquet)

total_rows = q_master.select(pl.len()).collect().item()
q_master = q_master.with_columns(
    pl.col("Timestamp").str.strptime(pl.Datetime, format="%Y/%m/%d %H:%M", strict=False).alias("ts")
)
q_valid = q_master.filter(pl.col("ts").is_not_null())
n = q_valid.select(pl.len()).collect().item()
print(f"Total: {total_rows:,} | Valid: {n:,}")

Total: 31,898,669 | Valid: 31,898,669


In [16]:
q_sorted = q_valid.sort("ts")
train_end_idx = int(n * 0.6)
val_end_idx = int(n * 0.8)

train_cut_ts = q_sorted.select("ts").slice(train_end_idx, 1).collect().item()
val_cut_ts = q_sorted.select("ts").slice(val_end_idx, 1).collect().item()
print("Train cutoff:", train_cut_ts, "| Val cutoff:", val_cut_ts)

q_split = q_sorted.with_columns(
    pl.when(pl.col("ts") <= train_cut_ts).then(pl.lit("train"))
      .when(pl.col("ts") <= val_cut_ts).then(pl.lit("val"))
      .otherwise(pl.lit("test")).alias("split")
)

print(q_split.group_by("split").agg(pl.len().alias("rows")).collect())
print(q_split.group_by(["split", "Is Laundering"]).agg(pl.len().alias("rows")).collect())

Train cutoff: 2022-09-09 20:43:00 | Val cutoff: 2022-09-14 05:46:00
shape: (3, 2)
┌───────┬──────────┐
│ split ┆ rows     │
│ ---   ┆ ---      │
│ str   ┆ u32      │
╞═══════╪══════════╡
│ train ┆ 19139731 │
│ test  ┆ 6378958  │
│ val   ┆ 6379980  │
└───────┴──────────┘
shape: (6, 3)
┌───────┬───────────────┬──────────┐
│ split ┆ Is Laundering ┆ rows     │
│ ---   ┆ ---           ┆ ---      │
│ str   ┆ i64           ┆ u32      │
╞═══════╪═══════════════╪══════════╡
│ test  ┆ 1             ┆ 10640    │
│ val   ┆ 1             ┆ 9054     │
│ val   ┆ 0             ┆ 6370926  │
│ test  ┆ 0             ┆ 6368318  │
│ train ┆ 1             ┆ 15536    │
│ train ┆ 0             ┆ 19124195 │
└───────┴───────────────┴──────────┘


# 3) Feature Engineering (gnn_gatv2_v2)


In [17]:
PAYMENT_RISK_MAP = {"ACH": 3.0, "Bitcoin": 2.5, "Cheque": 2.5, "Cash": 1.5, "Credit Card": 1.3, "Wire": 1.0, "Reinvestment": 1.0}
ENTITY_TYPE_SCORE_MAP = {"Corporation": 2.0, "Sole Proprietorship": 1.5}
HIGH_RISK_SENDER_BANKS = ["National Bank of Dallas","Savings Bank of Augusta","China Bank #27","India Bank #96","Brazil Bank #128","National Bank of Milford","Savings Bank of Sacramento","Saudi Arabia Bank #14","Israel Bank #16","Golden Credit Union"]
HIGH_RISK_RECEIVER_BANKS = ["China Bank #292","China Bank #27","China Bank #22","Japan Bank #143","Brazil Bank #50","Bank of Denver","Saudi Arabia Bank #56","Israel Bank #48","The Pine Bank","National Bank of Milford"]

In [18]:

q_feat = q_split.with_columns([
    pl.col("ts").dt.date().alias("ts_day"),
    pl.col("ts").dt.hour().cast(pl.Int16).alias("hour"),
    pl.col("ts").dt.weekday().cast(pl.Int8).alias("day_of_week"),
    pl.col("ts").dt.weekday().is_in([6, 7]).alias("is_weekend"),
    pl.col("ts").dt.hour().is_between(0, 5).alias("is_dawn"),
]).with_columns([
    (pl.col("hour").cast(pl.Float32) * (2 * np.pi / 24)).sin().alias("hour_sin"),
    (pl.col("hour").cast(pl.Float32) * (2 * np.pi / 24)).cos().alias("hour_cos"),
])

# Amount
q_feat = q_feat.with_columns([
    pl.col("Amount_Paid_USD").log1p().alias("log_amount_paid_usd"),
    pl.col("Amount_Received_USD").log1p().alias("log_amount_received_usd"),
    (pl.col("Amount_Paid_USD") % 1000 == 0).alias("is_round_1000_paid"),
    (pl.col("Amount_Received_USD") % 1000 == 0).alias("is_round_1000_received"),
    (pl.col("Amount_Paid_USD") % 10000 == 0).alias("is_round_10000_paid"),
])

# Payment Format
q_feat = q_feat.with_columns([
    (pl.col("Payment Format") == "ACH").alias("is_ach"),
    (pl.col("Payment Format") == "Cheque").alias("is_cheque"),
    (pl.col("Payment Format") == "Bitcoin").alias("is_bitcoin_fmt"),
    (pl.col("Payment Format") == "Cash").alias("is_cash"),
    (pl.col("Payment Format") == "Credit Card").alias("is_credit_card"),
    (pl.col("Payment Format") == "Wire").alias("is_wire"),
    (pl.col("Payment Format") == "Reinvestment").alias("is_reinvestment"),
    ((pl.col("Payment Format") == "Bitcoin") | (pl.col("Payment Currency") == "Bitcoin")).alias("is_crypto_transfer"),
    ((pl.col("Payment Format") == "ACH") & (pl.col("Amount_Paid_USD") >= 1_000_000)).alias("is_high_value_ach"),
    pl.col("Payment Format").replace(PAYMENT_RISK_MAP, default=1.0).cast(pl.Float32).alias("payment_method_risk"),
])
q_feat = q_feat.with_columns([
    (pl.col("payment_method_risk") * pl.col("log_amount_paid_usd")).alias("risk_x_log_paid"),
    (pl.col("is_ach").cast(pl.Int8) * pl.col("log_amount_paid_usd")).alias("ach_x_log_paid"),
    (pl.col("is_bitcoin_fmt").cast(pl.Int8) * pl.col("log_amount_paid_usd")).alias("btc_x_log_paid"),
])

# Entity + Bank
q_feat = q_feat.with_columns([
    pl.col("Sender_Entity").cast(pl.Utf8).str.replace(r"\s*#\d+.*$", "").alias("sender_entity_type"),
    pl.col("Receiver_Entity").cast(pl.Utf8).str.replace(r"\s*#\d+.*$", "").alias("receiver_entity_type"),
])
q_feat = q_feat.with_columns([
    pl.col("sender_entity_type").replace(ENTITY_TYPE_SCORE_MAP, default=1.0).cast(pl.Float32).alias("sender_entity_type_score"),
    pl.col("receiver_entity_type").replace(ENTITY_TYPE_SCORE_MAP, default=1.0).cast(pl.Float32).alias("receiver_entity_type_score"),
    pl.col("Sender_Bank_Name").is_in(HIGH_RISK_SENDER_BANKS).alias("high_risk_sender_bank_flag"),
    pl.col("Receiver_Bank_Name").is_in(HIGH_RISK_RECEIVER_BANKS).alias("high_risk_receiver_bank_flag"),
])

# Rolling window features (shifted by 1 to avoid leakage)
q_feat = q_feat.with_columns(pl.lit(1).cast(pl.Int32).alias("_one"))
q_feat = q_feat.sort(["From Account", "ts"])
q_feat = q_feat.with_columns([
    pl.col("_one").rolling_sum_by("ts", window_size="1h").shift(1).over("From Account").alias("s_cnt_1h_past"),
    pl.col("Amount_Paid_USD").rolling_sum_by("ts", window_size="1h").shift(1).over("From Account").alias("s_sum_paid_1h_past"),
    pl.col("Amount_Paid_USD").rolling_mean_by("ts", window_size="1h").shift(1).over("From Account").alias("s_mean_paid_1h_past"),
    pl.col("_one").rolling_sum_by("ts", window_size="24h").shift(1).over("From Account").alias("s_cnt_24h_past"),
])
q_feat = q_feat.sort(["To Account", "ts"])
q_feat = q_feat.with_columns([
    pl.col("_one").rolling_sum_by("ts", window_size="1h").shift(1).over("To Account").alias("r_cnt_1h_past"),
    pl.col("Amount_Received_USD").rolling_sum_by("ts", window_size="1h").shift(1).over("To Account").alias("r_sum_recv_1h_past"),
    pl.col("Amount_Received_USD").rolling_mean_by("ts", window_size="1h").shift(1).over("To Account").alias("r_mean_recv_1h_past"),
    pl.col("_one").rolling_sum_by("ts", window_size="24h").shift(1).over("To Account").alias("r_cnt_24h_past"),
])

PAST_COLS = ["s_cnt_1h_past","s_sum_paid_1h_past","s_mean_paid_1h_past","s_cnt_24h_past",
             "r_cnt_1h_past","r_sum_recv_1h_past","r_mean_recv_1h_past","r_cnt_24h_past"] # 중복 제거일까
q_feat = q_feat.with_columns([pl.col(c).fill_null(0) for c in PAST_COLS])

print("Feature engineering done")
print(q_feat.select(pl.len()).collect())

Feature engineering done
shape: (1, 1)
┌──────────┐
│ len      │
│ ---      │
│ u32      │
╞══════════╡
│ 31898669 │
└──────────┘


# 4) Collect + Node Mapping + Edge Feature Preparation


In [19]:

EDGE_FEAT_COLS = [
    "is_weekend","is_dawn","hour_sin","hour_cos",
    "log_amount_paid_usd","log_amount_received_usd",
    "is_round_1000_paid","is_round_1000_received","is_round_10000_paid",
    "is_ach","is_cheque","is_bitcoin_fmt","is_cash","is_credit_card","is_wire","is_reinvestment",
    "is_crypto_transfer","is_high_value_ach","payment_method_risk",
    "risk_x_log_paid","ach_x_log_paid","btc_x_log_paid",
    "sender_entity_type_score","receiver_entity_type_score",
    "high_risk_sender_bank_flag","high_risk_receiver_bank_flag",
    "s_cnt_1h_past","s_sum_paid_1h_past","s_mean_paid_1h_past","s_cnt_24h_past",
    "r_cnt_1h_past","r_sum_recv_1h_past","r_mean_recv_1h_past","r_cnt_24h_past",
]
META_COLS = ["From Account","To Account","Is Laundering","split","ts_day","ts", "Pattern_Type", "Pattern_Detail"]

select_cols = META_COLS + EDGE_FEAT_COLS
select_cols = [c for c in select_cols if c in q_feat.columns]

print("Collecting data...")
df_all = q_feat.select(select_cols).collect().to_pandas()
df_all['ts_int'] = df_all['ts'].astype('int64') // 10**9
print(f"Collected: {df_all.shape}")

# Bool -> float
for c in EDGE_FEAT_COLS:
    if c in df_all.columns and df_all[c].dtype == bool:
        df_all[c] = df_all[c].astype(float)
    if c in df_all.columns:
        df_all[c] = df_all[c].fillna(0).astype(np.float32)

# Split
df_train = df_all[df_all["split"] == "train"].copy()
df_val = df_all[df_all["split"] == "val"].copy()
df_test = df_all[df_all["split"] == "test"].copy()

LABEL_COL = "Is Laundering"
for tag, df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    pos = df[LABEL_COL].sum()
    print(f"{tag:5s} shape={df.shape}  pos={int(pos):,}  rate={pos/len(df):.6f}")

Collected: (31898669, 43)
Train shape=(19139731, 43)  pos=15,536  rate=0.000812
Val   shape=(6379980, 43)  pos=9,054  rate=0.001419
Test  shape=(6378958, 43)  pos=10,640  rate=0.001668


In [20]:

# Node mapping
all_accounts = pd.concat([
    df_train["From Account"], df_train["To Account"],
    df_val["From Account"], df_val["To Account"],
    df_test["From Account"], df_test["To Account"],
]).unique()
account_map = {acc: i for i, acc in enumerate(all_accounts)}
num_nodes = len(all_accounts)
print(f"Unique nodes: {num_nodes:,}")

for df in [df_train, df_val, df_test]:
    df["src"] = df["From Account"].map(account_map)
    df["dst"] = df["To Account"].map(account_map)

Unique nodes: 2,076,999


In [21]:

# Edge features: RobustScaler (fit on train only)
feature_cols = [c for c in EDGE_FEAT_COLS if c in df_train.columns]
scaler = RobustScaler()
X_train = scaler.fit_transform(df_train[feature_cols].values)
X_val = scaler.transform(df_val[feature_cols].values)
X_test = scaler.transform(df_test[feature_cols].values)

edge_time_train = torch.tensor(df_train["ts_int"].values, dtype=torch.long)
edge_time_val = torch.tensor(df_val["ts_int"].values, dtype=torch.long)
edge_time_test = torch.tensor(df_test["ts_int"].values, dtype=torch.long)

edge_attr_train = torch.tensor(X_train, dtype=torch.float)
edge_attr_val = torch.tensor(X_val, dtype=torch.float)
edge_attr_test = torch.tensor(X_test, dtype=torch.float)
y_train = torch.tensor(df_train[LABEL_COL].values, dtype=torch.float)
y_val = torch.tensor(df_val[LABEL_COL].values, dtype=torch.float)
y_test = torch.tensor(df_test[LABEL_COL].values, dtype=torch.float)

edge_index_train = torch.tensor(np.stack([df_train["src"].values, df_train["dst"].values]), dtype=torch.long)
edge_index_val = torch.tensor(np.stack([df_val["src"].values, df_val["dst"].values]), dtype=torch.long)
edge_index_test = torch.tensor(np.stack([df_test["src"].values, df_test["dst"].values]), dtype=torch.long)

num_edge_features = edge_attr_train.shape[1]
print(f"Edge features: {num_edge_features}")

Edge features: 34


# 5) Node Features (계좌 통계 집계)


In [22]:
def create_node_features_simple(df_train, all_accounts):
    """Train 데이터만으로 노드 피처 생성 (데이터 누수 방지)"""
    amt_cols = [c for c in ["log_amount_paid_usd", "log_amount_received_usd",
                            "Amount_Paid_USD", "Amount_Received_USD"]
                if c in df_train.columns]
    amt_cols = [c for c in amt_cols if c != LABEL_COL]
    print(f"  Aggregation columns: {amt_cols}")

    out_stats = df_train.groupby("From Account")[amt_cols].agg(["sum", "mean", "count", "max"])
    out_stats.columns = [f"out_{c}_{s}" for c, s in out_stats.columns]

    in_stats = df_train.groupby("To Account")[amt_cols].agg(["sum", "mean", "count", "max"])
    in_stats.columns = [f"in_{c}_{s}" for c, s in in_stats.columns]

    node_df = pd.DataFrame(index=all_accounts)
    node_df = node_df.join(out_stats, how="left").join(in_stats, how="left").fillna(0)

    print(f"  Node features (train only, no leakage): {node_df.shape}")
    return node_df

node_df = create_node_features_simple(df_train, all_accounts)
node_scaler = RobustScaler()
x = torch.tensor(node_scaler.fit_transform(node_df.values), dtype=torch.float)
print(f"Node feature tensor: {x.shape}")

  Aggregation columns: ['log_amount_paid_usd', 'log_amount_received_usd']
  Node features (train only, no leakage): (2076999, 16)
Node feature tensor: torch.Size([2076999, 16])


# 6) Temporal Graph (Data Leakage 방지)


In [23]:
n_train = len(df_train)
n_val = len(df_val)
n_test = len(df_test)

data_train = Data(x=x, edge_index=edge_index_train, edge_attr=edge_attr_train, y=y_train, num_nodes=num_nodes)

data_val = Data(x=x,
    edge_index=torch.cat([edge_index_train, edge_index_val], dim=1),
    edge_attr=torch.cat([edge_attr_train, edge_attr_val], dim=0),
    y=torch.cat([y_train, y_val]), num_nodes=num_nodes)

data_test = Data(x=x,
    edge_index=torch.cat([edge_index_train, edge_index_val, edge_index_test], dim=1),
    edge_attr=torch.cat([edge_attr_train, edge_attr_val, edge_attr_test], dim=0),
    y=torch.cat([y_train, y_val, y_test]), num_nodes=num_nodes)

val_mask = torch.zeros(n_train + n_val, dtype=torch.bool)
val_mask[n_train:] = True
test_mask = torch.zeros(n_train + n_val + n_test, dtype=torch.bool)
test_mask[n_train + n_val:] = True

val_target_edge_attr = edge_attr_val
test_target_edge_attr = edge_attr_test

print("Train graph:", data_train)
print("Val   graph:", data_val)
print("Test  graph:", data_test)


Train graph: Data(x=[2076999, 16], edge_index=[2, 19139731], edge_attr=[19139731, 34], y=[19139731], num_nodes=2076999)
Val   graph: Data(x=[2076999, 16], edge_index=[2, 25519711], edge_attr=[25519711, 34], y=[25519711], num_nodes=2076999)
Test  graph: Data(x=[2076999, 16], edge_index=[2, 31898669], edge_attr=[31898669, 34], y=[31898669], num_nodes=2076999)


# 7) Model: EdgeClassifier + BiEdgeWeightedConv


In [24]:
ACT_FN = {'relu': nn.ReLU, 'elu': nn.ELU, 'gelu': nn.GELU}

class EdgeWeightedConv(MessagePassing):
    def __init__(self, dim, edge_dim):
        super().__init__(aggr='mean')
        self.lin_neigh = nn.Linear(dim, dim)
        self.lin_self = nn.Linear(dim, dim)
        self.edge_gate = nn.Sequential(nn.Linear(edge_dim, dim), nn.Sigmoid())

    def forward(self, x, edge_index, edge_attr=None):
        aggr = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        return self.lin_self(x) + aggr

    def message(self, x_j, edge_attr):
        h = self.lin_neigh(x_j)
        if edge_attr is not None:
            h = h * self.edge_gate(edge_attr)
        return h

class BiEdgeWeightedConv(nn.Module):
    def __init__(self, dim, edge_dim):
        super().__init__()
        self.conv_in = EdgeWeightedConv(dim, edge_dim)
        self.conv_out = EdgeWeightedConv(dim, edge_dim)
        self.combine = nn.Linear(2 * dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.act = nn.ReLU()

    def forward(self, x, edge_index, edge_attr=None):
        h_in = self.conv_in(x, edge_index, edge_attr)
        rev = torch.stack([edge_index[1], edge_index[0]], dim=0)
        h_out = self.conv_out(x, rev, edge_attr)
        return self.act(self.norm(self.combine(torch.cat([h_in, h_out], dim=-1))))

# EdgeClassifier.__init__에서 변경
# 기존: self.node_emb = nn.Embedding(num_nodes, emb_dim)
# 변경: node feature만 사용

class EdgeClassifier(nn.Module):
    def __init__(self, num_nodes, hidden_dim, edge_dim, num_gnn_layers=2,
             dropout=0.3, node_feat_dim=16,
             clf_layers=2, activation='relu'):
        super().__init__()
        self.dropout_rate = dropout
        self.edge_dim = edge_dim
        act_cls = ACT_FN.get(activation, nn.ReLU)

        # ★ nn.Embedding 제거! node feature projection만 사용
        self.node_proj = nn.Sequential(
            nn.Linear(node_feat_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            act_cls()
        )

        self.gnn_layers = nn.ModuleList()
        self.gnn_norms = nn.ModuleList()
        for _ in range(num_gnn_layers):
            self.gnn_layers.append(BiEdgeWeightedConv(hidden_dim, edge_dim=edge_dim))
            self.gnn_norms.append(nn.LayerNorm(hidden_dim))

        clf_in = 2 * hidden_dim + edge_dim
        clf = [nn.Linear(clf_in, hidden_dim), nn.BatchNorm1d(hidden_dim), act_cls(), nn.Dropout(dropout)]
        for _ in range(clf_layers - 1):
            clf += [nn.Linear(hidden_dim, hidden_dim // 2), act_cls(), nn.Dropout(dropout)]
            hidden_dim = hidden_dim // 2
        clf.append(nn.Linear(hidden_dim, 1))
        self.classifier = nn.Sequential(*clf)

    def encode_nodes(self, n_id, edge_index, edge_attr=None, x=None):
        # ★ node feature만 사용 (Embedding 없음)
        h = self.node_proj(x)
        for i, (conv, norm) in enumerate(zip(self.gnn_layers, self.gnn_norms)):
            h_new = norm(conv(h, edge_index, edge_attr=edge_attr))
            if i < len(self.gnn_layers) - 1:
                h_new = F.relu(h_new)
                h_new = F.dropout(h_new, p=self.dropout_rate, training=self.training)
            h = h + h_new
        return h

    def forward(self, batch, target_edge_feat=None):
        h = self.encode_nodes(batch.n_id, batch.edge_index, batch.edge_attr, getattr(batch, 'x', None))
        src = h[batch.edge_label_index[0]]
        dst = h[batch.edge_label_index[1]]
        ef = target_edge_feat if target_edge_feat is not None else batch.edge_attr[:batch.edge_label.size(0)]
        return self.classifier(torch.cat([src, dst, ef], dim=1)).squeeze(-1)


# 8) Training & Evaluation Functions


In [25]:

@torch.no_grad()
def evaluate(model, loader, device, target_edge_attr_lookup=None):
    model.eval()
    ys, ps = [], []
    for batch in loader:
        batch = batch.to(device)
        if target_edge_attr_lookup is not None and hasattr(batch, "input_id"):
            tef = target_edge_attr_lookup[batch.input_id.cpu()].to(device)
        else:
            tef = None
        pred = torch.sigmoid(model(batch, tef))
        ys.append(batch.edge_label.cpu().numpy())
        ps.append(pred.cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

def safe_auc(y, p):
    if len(np.unique(y.astype(int))) <= 1:
        return np.nan, np.nan
    return roc_auc_score(y, p), average_precision_score(y, p)

def workload_to_find_k_positives(y_true, y_score, k_pos):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)
    total_pos = y_true.sum()
    if total_pos == 0:
        return 0, 0.0, 0.0, 0.0, 0
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    cum_tp = np.cumsum(y_sorted)
    target = min(k_pos, int(total_pos))
    idx = np.searchsorted(cum_tp, target, side="left")
    N_required = int(idx + 1)
    k_found = int(cum_tp[idx])
    precision = k_found / (N_required + 1e-12)
    recall = k_found / (total_pos + 1e-12)
    f1 = (2 * k_found) / (2 * k_found + (N_required - k_found) + (total_pos - k_found) + 1e-12)
    return N_required, precision, recall, f1, k_found

# 09) 외부 정의 함수

In [26]:
# Top-k 로직 교체용 함수:“k개 적발하려면 몇 건을 봐야 하는지”를 계산하는 함수
def workload_to_find_k_positives(y_true, y_score, k_pos):
    """
    목표: true positive를 k_pos개 '찾기' 위해
         상위 점수부터 몇 건(N)을 조사해야 하는지 계산

    Returns:
      N_required: 필요한 조사 건수
      precision:  k_found / N_required
      recall:     k_found / total_pos
      f1:         f1 (top-N을 양성으로 가정한 경우)
      k_found:    실제로 찾은 TP 수 (total_pos < k_pos면 total_pos)
    """
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)

    total_pos = y_true.sum()
    if total_pos == 0:
        return 0, 0.0, 0.0, 0.0, 0

    # 점수 내림차순 정렬
    order = np.argsort(-y_score)
    y_sorted = y_true[order]

    # 누적 TP
    cum_tp = np.cumsum(y_sorted)

    target = min(k_pos, int(total_pos))

    # cum_tp >= target 되는 첫 index
    idx = np.searchsorted(cum_tp, target, side="left")
    N_required = int(idx + 1)  # index -> count

    k_found = int(cum_tp[idx])  # 보통 target과 같음(동점/중복 없으면)

    precision = k_found / (N_required + 1e-12)
    recall = k_found / (total_pos + 1e-12)

    # top-N을 양성으로 보면: TP=k_found, FP=N-TP, FN=total_pos-TP
    fp = N_required - k_found
    fn = total_pos - k_found
    f1 = (2 * k_found) / (2 * k_found + fp + fn + 1e-12)

    return N_required, precision, recall, f1, k_found

In [27]:
def per_day_workload_summary(df, day_col, label_col, score_col, k_pos_list=(30,50,100,200)):
    """
    df: pandas DataFrame with [day_col, label_col, score_col]
    day별로 'k_pos개 적발하기 위해 필요한 조사량 N' 계산 후
    k_pos마다 daily 분포 + mean/median/p90 요약을 반환
    """
    out_rows = []
    for day, g in df.groupby(day_col):
        y = g[label_col].astype(int).values
        s = g[score_col].values
        total_pos = int(y.sum())
        n = len(g)

        for k_pos in k_pos_list:
            N_req, p, r, f1, found = workload_to_find_k_positives(y, s, k_pos)
            out_rows.append({
                "day": day,
                "n_rows": n,
                "pos": total_pos,
                "k_pos": k_pos,
                "N_required": N_req,
                "precision_at_Nk": p,
                "recall_at_Nk": r,
                "f1_at_Nk": f1,
                "found_tp": found,
                "target_k": min(k_pos, total_pos),
                "hit_target": (found >= min(k_pos, total_pos)) and (total_pos > 0)
            })

    daily = pd.DataFrame(out_rows)

    # day에 pos=0이면 N_required=0이 되고 의미가 약하니,
    # 운영 KPI로는 "pos>0인 day"만 요약하는게 보통 더 깔끔함
    daily_pos = daily[daily["pos"] > 0].copy()

    def summarize(sub):
        # N_required는 "작을수록 좋음"
        return pd.Series({
            "days": sub["day"].nunique(),
            "mean_N_required": sub["N_required"].mean(),
            "median_N_required": sub["N_required"].median(),
            "p90_N_required": sub["N_required"].quantile(0.90),
            "mean_precision": sub["precision_at_Nk"].mean(),
            "median_precision": sub["precision_at_Nk"].median(),
            "p90_precision": sub["precision_at_Nk"].quantile(0.90),
        })

    summary = (
        daily_pos
        .groupby("k_pos", as_index=False)
        .apply(summarize)
        .reset_index(drop=True)
    )

    return daily, summary

In [28]:
def log_per_day_kpi_to_wandb(split_name, df_raw_with_day, y_true, y_score, k_list=(30,50,100,200)):
    """
    split_name: "val" or "test"
    df_raw_with_day: pandas DF that includes column "ts_day" aligned with y_true/y_score index
    y_true/y_score: numpy-like aligned arrays
    """
    tmp = pd.DataFrame({
        "ts_day": df_raw_with_day["ts_day"].values,
        "label": np.asarray(y_true).astype(int),
        "score": np.asarray(y_score),
    })

    daily, summary = per_day_workload_summary(
        tmp,
        day_col="ts_day",
        label_col="label",
        score_col="score",
        k_pos_list=k_list
    )

    # (A) 요약 로그: k별 mean/median/p90
    for _, row in summary.iterrows():
        k = int(row["k_pos"])
        wandb.log({
            f"{split_name}_perday_find_{k}_mean_N": float(row["mean_N_required"]),
            f"{split_name}_perday_find_{k}_median_N": float(row["median_N_required"]),
            f"{split_name}_perday_find_{k}_p90_N": float(row["p90_N_required"]),
            f"{split_name}_perday_find_{k}_mean_precision": float(row["mean_precision"]),
            f"{split_name}_perday_find_{k}_median_precision": float(row["median_precision"]),
            f"{split_name}_perday_find_{k}_p90_precision": float(row["p90_precision"]),
            f"{split_name}_perday_days_with_pos_{k}": int(row["days"]),
        })

    # (B) 분포 확인용 테이블(너무 크면 상위 일부만)
    # day*k 만큼 행이 생김 -> 기간 짧으면 OK, 길면 샘플링/요약만 남기기
    table = wandb.Table(dataframe=daily.head(5000))
    wandb.log({f"{split_name}_perday_workload_table": table})

In [29]:
def topN_distinct_account_metrics(df_meta, y_true, y_score, topN,
                                  account_cols=("From Account","To Account"),
                                  score_agg="max"):
    """
    전체 기간에서 score 내림차순 topN '거래'를 뽑고,
    그 안에서 account_cols(From/To) 기준 distinct 계좌로 묶어 평가.

    label: 계좌가 topN 거래 중 laundering 거래가 1번이라도 있으면 1 (max)
    score: 계좌 내 score 집계 (max 추천)
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(y_score)

    tmp = df_meta[list(account_cols)].copy()
    tmp["label_trx"] = y
    tmp["score_trx"] = s

    # topN 거래 선택
    tmp = tmp.sort_values("score_trx", ascending=False).head(int(min(topN, len(tmp))))

    # 거래 -> 계좌 집계
    agg = aggregate_to_account_level(
        df_meta=tmp, y_true=tmp["label_trx"].values, y_score=tmp["score_trx"].values,
        account_cols=account_cols, score_agg=score_agg
    )

    y_acc = agg["label"].values
    tp = int((y_acc == 1).sum())
    fp = int((y_acc == 0).sum())
    precision = tp / (tp + fp + 1e-12)

    return {
        "topN_trx": int(min(topN, len(tmp))),
        "distinct_accounts": int(len(agg)),
        "tp_accounts": tp,
        "fp_accounts": fp,
        "precision_accounts": float(precision),
    }

In [30]:
def score_bin_distribution(y_true, y_score, n_bins=10, score_scale=1000):
    """
    score를 0~score_scale로 스케일링한 뒤,
    bin별 label(0/1) 건수 집계.
    항상 normal_cnt / laundering_cnt 컬럼이 존재하도록 보정.
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(y_score)

    s_scaled = np.clip(s * score_scale, 0, score_scale)
    bins = np.linspace(0, score_scale, n_bins + 1)

    # 0..n_bins-1
    bin_idx = np.digitize(s_scaled, bins, right=False) - 1
    bin_idx = np.clip(bin_idx, 0, n_bins - 1)

    df = pd.DataFrame({"bin": bin_idx, "label": y})

    # label별 count (항상 0/1 둘 다 컬럼을 만들도록 reindex)
    cnt = (
        df.groupby(["bin", "label"])
          .size()
          .unstack(fill_value=0)
          .reindex(columns=[0, 1], fill_value=0)   # ✅ 핵심
          .rename(columns={0: "normal_cnt", 1: "laundering_cnt"})
          .reset_index()
    )

    # bin_range 붙이기
    cnt["bin_range"] = cnt["bin"].apply(lambda i: f"{int(bins[i])}-{int(bins[i+1])}")

    # 보기 좋게 정렬
    cnt = cnt[["bin", "bin_range", "normal_cnt", "laundering_cnt"]]
    return cnt

In [31]:
def aggregate_to_account_level(df_meta, y_true, y_score,
                               account_cols=("From Account",),
                               score_agg="max"):
    """
    거래 단위 (y_true, y_score)를 계좌 단위로 집계.
    account_cols: ("From Account",) or ("To Account",) or ("From Account","To Account")
    score_agg: "max" (추천) or "mean"
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(y_score)

    # (A) 거래 메타 + label/score 결합
    tmp = df_meta[list(account_cols)].copy()
    tmp["label_trx"] = y
    tmp["score_trx"] = s

    # (B) From/To 둘 다 쓰는 경우: long 형태로 펼치기
    if len(account_cols) == 2:
        a1, a2 = account_cols
        tmp1 = tmp[[a1, "label_trx", "score_trx"]].rename(columns={a1: "account"})
        tmp2 = tmp[[a2, "label_trx", "score_trx"]].rename(columns={a2: "account"})
        tmp_long = pd.concat([tmp1, tmp2], axis=0, ignore_index=True)
    else:
        a1 = account_cols[0]
        tmp_long = tmp[[a1, "label_trx", "score_trx"]].rename(columns={a1: "account"})

    # 결측 계좌 방어
    tmp_long = tmp_long.dropna(subset=["account"])

    # (C) 계좌 단위 집계
    if score_agg == "mean":
        agg = tmp_long.groupby("account").agg(
            label=("label_trx", "max"),
            score=("score_trx", "mean"),
            n_hits=("score_trx", "size"),
        ).reset_index()
    else:  # "max"
        agg = tmp_long.groupby("account").agg(
            label=("label_trx", "max"),
            score=("score_trx", "max"),
            n_hits=("score_trx", "size"),
        ).reset_index()

    return agg  # columns: account, label, score, n_hits

In [32]:
def log_per_day_account_kpi_to_wandb(split_name, df_meta, y_true, y_score,
                                     k_list=(30,50,100,200),
                                     account_cols=("From Account","To Account"),
                                     score_agg="max"):
    """
    하루마다 (거래->계좌 집계) 후 workload_to_find_k_positives를 계좌단위로 계산.
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(y_score)

    tmp = df_meta[["ts_day"] + list(account_cols)].copy()
    tmp["label_trx"] = y
    tmp["score_trx"] = s

    rows = []
    for day, g in tmp.groupby("ts_day"):
        agg = aggregate_to_account_level(
            df_meta=g, y_true=g["label_trx"].values, y_score=g["score_trx"].values,
            account_cols=account_cols, score_agg=score_agg
        )
        y_acc = agg["label"].values
        s_acc = agg["score"].values

        for k_pos in k_list:
            N_req, p, r, f1, found = workload_to_find_k_positives(y_acc, s_acc, k_pos)
            rows.append({
                "ts_day": day,
                "k_pos": k_pos,
                "n_accounts": len(agg),
                "pos_accounts": int(y_acc.sum()),
                "N_required": N_req,
                "precision": p,
                "recall": r,
                "f1": f1,
                "found_tp": found
            })

    daily = pd.DataFrame(rows)
    daily_pos = daily[daily["pos_accounts"] > 0].copy()

    # 요약 로그
    if len(daily_pos) > 0:
        summ = daily_pos.groupby("k_pos").agg(
            days=("ts_day", "nunique"),
            mean_N=("N_required", "mean"),
            median_N=("N_required", "median"),
            p90_N=("N_required", lambda x: x.quantile(0.9)),
            mean_precision=("precision", "mean"),
        ).reset_index()

        for _, r in summ.iterrows():
            k = int(r["k_pos"])
            wandb.log({
                f"{split_name}_acc_perday_find_{k}_mean_N": float(r["mean_N"]),
                f"{split_name}_acc_perday_find_{k}_median_N": float(r["median_N"]),
                f"{split_name}_acc_perday_find_{k}_p90_N": float(r["p90_N"]),
                f"{split_name}_acc_perday_find_{k}_mean_precision": float(r["mean_precision"]),
                f"{split_name}_acc_perday_days_with_pos_{k}": int(r["days"]),
            })

    wandb.log({f"{split_name}_acc_perday_table": wandb.Table(dataframe=daily.head(5000))})

In [33]:
def make_trx_scores_from_node_probs(edges_meta_df: pd.DataFrame,
                                   node_prob: np.ndarray,
                                   src_col="src_id", dst_col="dst_id",
                                   agg="max") -> np.ndarray:
    """
    edges_meta_df에 src_id/dst_id가 있다고 가정.
    node_prob: shape [num_nodes] (numpy)
    agg: "max" | "mean"
    """
    src = edges_meta_df[src_col].to_numpy().astype(int)
    dst = edges_meta_df[dst_col].to_numpy().astype(int)

    ps = node_prob[src]
    pd_ = node_prob[dst]

    if agg == "mean":
        return 0.5 * (ps + pd_)
    return np.maximum(ps, pd_)

In [34]:
def get_pattern_analysis_tables(df, prob, thr):
    """
    Pandas 기반으로 패턴별/상세별 Recall을 계산하여 WandB Table 2개를 반환
    """

    # 1. 실제 세탁 거래(1)만 추출하여 분석용 DF 생성
    analysis = pd.DataFrame({
        "Pattern_Type": df["Pattern_Type"],
        "Pattern_Detail": df["Pattern_Detail"],
        "y_true": df["Is Laundering"],
        "is_detected": (prob >= thr).astype(int)
    })
    pattern_df = analysis[analysis["y_true"] == 1].copy()

    if pattern_df.empty:
        return wandb.Table(columns=["Pattern_Type", "Recall_Pct"]), wandb.Table(columns=["Pattern_Type", "Pattern_Detail", "Recall_Pct"])

    # 2. 패턴별 요약 (Summary)
    summary = pattern_df.groupby("Pattern_Type").agg(
        Total_Actual=("is_detected", "count"),
        Detected_Count=("is_detected", "sum")
    ).reset_index()
    summary["Recall_Pct"] = (summary["Detected_Count"] / summary["Total_Actual"] * 100).round(2)
    summary = summary.sort_values("Recall_Pct") # 잘 못잡는 순서 정렬

    # 3. 상세 영향도 (Detail)
    detail = pattern_df[pattern_df["Pattern_Detail"] != ""].groupby(["Pattern_Type", "Pattern_Detail"]).agg(
        Total_Actual=("is_detected", "count"),
        Detected_Count=("is_detected", "sum")
    ).reset_index()
    if not detail.empty:
        detail["Recall_Pct"] = (detail["Detected_Count"] / detail["Total_Actual"] * 100).round(2)
        detail = detail.sort_values(["Pattern_Type", "Recall_Pct"])

    return wandb.Table(dataframe=summary), wandb.Table(dataframe=detail)

In [ ]:
def get_pattern_analysis_tables_find_k(df, prob, k_pos_target):
    """
    Find-K 로직 기반으로 패턴별/상세별 Recall을 계산하여 WandB Table 2개를 반환.
    thr 대신 k_pos_target(예: 3000)을 사용하여 상위 N개 조사 범위를 결정.
    """
    y_true = df["Is Laundering"].astype(int).values
    y_score = np.asarray(prob)

    # 1. 외부 함수를 사용하여 k_pos_target개를 찾기 위한 N_required 계산
    # (이미 정의하신 workload_to_find_k_positives 함수를 활용합니다)
    N_req, _, _, _, _ = workload_to_find_k_positives(y_true, y_score, k_pos_target)

    # 2. 분석용 데이터프레임 생성
    analysis = pd.DataFrame({
        "Pattern_Type": df["Pattern_Type"],
        "Pattern_Detail": df["Pattern_Detail"],
        "y_true": y_true,
        "score": y_score
    })

    # 점수 순 정렬 후 상위 N_req개만 'is_detected'로 표시
    analysis = analysis.sort_values("score", ascending=False).reset_index(drop=True)
    analysis["is_detected"] = 0
    analysis.loc[:N_req-1, "is_detected"] = 1

    # 3. 실제 세탁 거래(1)만 추출하여 패턴 분석 진행
    pattern_df = analysis[analysis["y_true"] == 1].copy()

    if pattern_df.empty:
        return (wandb.Table(columns=["Pattern_Type", "Total_Actual", "Detected_Count", "Recall_Pct"]),
                wandb.Table(columns=["Pattern_Type", "Pattern_Detail", "Total_Actual", "Detected_Count", "Recall_Pct"]))

    # 4. 패턴별 요약 (Summary)
    summary = pattern_df.groupby("Pattern_Type").agg(
        Total_Actual=("is_detected", "count"),
        Detected_Count=("is_detected", "sum")
    ).reset_index()
    summary["Recall_Pct"] = (summary["Detected_Count"] / summary["Total_Actual"] * 100).round(2)
    summary = summary.sort_values("Recall_Pct") # 검출률 낮은 순 정렬

    # 5. 상세 영향도 (Detail)
    detail = pattern_df[pattern_df["Pattern_Detail"] != ""].groupby(["Pattern_Type", "Pattern_Detail"]).agg(
        Total_Actual=("is_detected", "count"),
        Detected_Count=("is_detected", "sum")
    ).reset_index()

    if not detail.empty:
        detail["Recall_Pct"] = (detail["Detected_Count"] / detail["Total_Actual"] * 100).round(2)
        detail = detail.sort_values(["Pattern_Type", "Recall_Pct"])

    return wandb.Table(dataframe=summary), wandb.Table(dataframe=detail)

In [36]:
def log_find_k_error_analysis(split_name, df, y_true, y_score, k_pos_target, top_features):
    """
    find-K 로직과 호환되는 에러 분석:
    실제 Positive를 k_pos_target개 찾기 위해 필요한 상위 N개 구간을 정해 분석함.
    """
    y_true_arr = np.asarray(y_true).astype(int)
    y_score_arr = np.asarray(y_score)

    # 1. 외부함수 활용하여 필요한 조사 건수(N_req) 계산
    N_req, precision, recall, f1, k_found = workload_to_find_k_positives(y_true_arr, y_score_arr, k_pos_target)

    # 2. 분석용 DF 생성
    analysis_df = df.copy()
    analysis_df['prob'] = y_score_arr
    analysis_df['label'] = y_true_arr

    # 점수 내림차순 정렬 후 상위 N_req개에 대해 pred=1 부여
    analysis_df = analysis_df.sort_values("prob", ascending=False).reset_index(drop=True)
    analysis_df['pred'] = 0
    analysis_df.loc[:N_req-1, 'pred'] = 1  # 0부터 N_req건까지 1 부여

    # 3. TP/FP/FN 분류 (N_req 구간 기준)
    # TP: 상위 N개 중 실제 세탁 (k_found개)
    # FP: 상위 N개 중 실제 정상 (N_req - k_found개)
    # FN: 상위 N개 밖에 있는 실제 세탁 (total_pos - k_found개)
    def categorize(row):
        if row['label'] == 1 and row['pred'] == 1: return 'TP (정탐)'
        if row['label'] == 0 and row['pred'] == 1: return 'FP (오탐)'
        if row['label'] == 1 and row['pred'] == 0: return 'FN (미탐)'
        return 'TN'

    analysis_df['error_cat'] = analysis_df.apply(categorize, axis=1)

    # --- [A] 일자별 분석 (Trend) ---
    # find-K를 위해 조사한 N개 내에서의 일자별 정탐/오탐/미탐 분포
    error_by_day = analysis_df.groupby(['ts_day', 'error_cat']).size().unstack(fill_value=0).reset_index()
    wandb.log({f"{split_name}_find_{k_pos_target}_daily_error_trend": wandb.Table(dataframe=error_by_day)})

    # --- [B] 피처별 분석 (Characteristics) ---
    feat_stats = []
    for cat in ['TP (정탐)', 'FP (오탐)', 'FN (미탐)']:
        sub = analysis_df[analysis_df['error_cat'] == cat]
        if sub.empty: continue

        stat_row = {'category': cat, 'count': len(sub)}
        for f in top_features:
            if f not in sub.columns: continue
            val = sub[f]
            if pd.api.types.is_numeric_dtype(val):
                stat_row[f"{f}_mean"] = round(val.mean(), 4)
            else:
                stat_row[f"{f}_top_mode"] = str(val.mode()[0]) if not val.mode().empty else "N/A"
        feat_stats.append(stat_row)

    wandb.log({f"{split_name}_find_{k_pos_target}_feat_stats": wandb.Table(dataframe=pd.DataFrame(feat_stats))})

    # --- [C] 패턴 중심 분석 ---
    # 미탐(FN): 3000개를 잡기 위해 조사해야 하는 우선순위 밖으로 밀려난 패턴들
    fn_df = analysis_df[analysis_df['error_cat'] == 'FN (미탐)']
    if not fn_df.empty:
        fn_pattern = fn_df.groupby(['ts_day', 'Pattern_Type']).size().reset_index(name='fn_count')
        wandb.log({f"{split_name}_find_{k_pos_target}_missed_patterns_by_day": wandb.Table(dataframe=fn_pattern)})

        fn_detail = fn_df.groupby(['Pattern_Type', 'Pattern_Detail']).size().reset_index(name='fn_count')
        wandb.log({f"{split_name}_find_{k_pos_target}_missed_patterns_detail": wandb.Table(dataframe=fn_detail)})

    # 오탐(FP): 3000개를 잡는 과정에서 조사가 불가피했던 정상 패턴들
    fp_df = analysis_df[analysis_df['error_cat'] == 'FP (오탐)']
    if not fp_df.empty:
        fp_pattern = fp_df.groupby(['ts_day', 'Pattern_Type']).size().reset_index(name='fp_count')
        wandb.log({f"{split_name}_find_{k_pos_target}_false_alarm_patterns_by_day": wandb.Table(dataframe=fp_pattern)})

# 10) Best Model Retraining + Full Evaluation


In [ ]:
torch.cuda.empty_cache(); gc.collect()
seed_everything(1)

# ═══ 하이퍼파라미터 ═══
hidden_dim     = 256
num_gnn_layers = 3
dropout        = 0.1467
lr             = 0.000982
batch_size     = 8192
num_neighbors  = [10, 5]
grad_clip      = 1.494
clf_layers     = 2
activation     = 'elu'
weight_decay   = 0.00473

train_loader = LinkNeighborLoader(
    data_train, num_neighbors=num_neighbors, batch_size=batch_size,
    edge_label_index=data_train.edge_index, edge_label=data_train.y,
    shuffle=True, num_workers=0)

val_loader = LinkNeighborLoader(
    data_val, num_neighbors=num_neighbors, batch_size=batch_size,
    edge_label_index=data_val.edge_index[:, val_mask], edge_label=data_val.y[val_mask],
    shuffle=False, num_workers=0)

test_loader = LinkNeighborLoader(
    data_test, num_neighbors=num_neighbors, batch_size=batch_size,
    edge_label_index=data_test.edge_index[:, test_mask], edge_label=data_test.y[test_mask],
    shuffle=False, num_workers=0)


# ── Model ──
model = EdgeClassifier(
    num_nodes=num_nodes, hidden_dim=hidden_dim, edge_dim=num_edge_features,
    num_gnn_layers=num_gnn_layers, dropout=dropout, node_feat_dim=x.shape[1],
    clf_layers=clf_layers, activation=activation,
).to(device)

pw = 1.0
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=device))
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)

num_epochs = 60
patience = 15
best_v_ap = 0.0
best_epoch = 0
best_state = None

for epoch in range(num_epochs):
    model.train()
    tot_loss, n_batch = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}",
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} {postfix}')
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad(set_to_none=True)
        out = model(batch)
        loss = criterion(out, batch.edge_label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        tot_loss += loss.item()
        n_batch += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    scheduler.step()

    # Evaluate
    y_v, p_v = evaluate(model, val_loader, device, val_target_edge_attr)
    p_v = np.nan_to_num(p_v, nan=0.0)
    v_auc, v_ap = safe_auc(y_v, p_v)

    improved = ""
    if v_ap > best_v_ap:
        best_v_ap = v_ap
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())
        improved = " ★ NEW BEST"

    print(f"  Ep {epoch+1:2d} | loss={tot_loss/n_batch:.4f} | Val AUROC={v_auc:.4f} AUPRC={v_ap:.4f}{improved}")

    # Early stopping
    if (epoch + 1) - best_epoch >= patience:
        print(f"\n  ⏹ Early stop at epoch {epoch+1} (no improvement for {patience} epochs)")
        break

print(f"\n✅ Best Val AUPRC: {best_v_ap:.4f} (Epoch {best_epoch})")
model.load_state_dict(best_state)

# Final Test
y_t, p_t = evaluate(model, test_loader, device, test_target_edge_attr)
p_t = np.nan_to_num(p_t, nan=0.0)
t_auc, t_ap = safe_auc(y_t, p_t)
print(f"[Final Test] AUROC={t_auc:.4f}  AUPRC={t_ap:.4f}")

print("\n--- Workload ---")
for k in [450, 750, 1500, 3000]:
    N, p, r, f1, found = workload_to_find_k_positives(y_t, p_t, k)
    print(f"  k={k:5d}: N={N:6,} prec={p:.3f} rec={r:.3f} f1={f1:.3f}")

# # =======================================================
# # ★ GNN Permutation Importance 계산 (수동 지정 대체)
# # =======================================================
# print("\n📊 Calculating Permutation Importance for GNN...")

# # 0. feature_names가 없다면 강제로 생성 (오류 방지)
# # 만약 전처리 단계에서 저장한 리스트가 있다면 그것을 쓰세요.
# if 'feature_names' not in locals() and 'feature_names' not in globals():
#     # 예: "Amount", "Timestamp" 등 텐서의 열 순서와 일치해야 함
#     print("[WARN] feature_names를 찾을 수 없어 기본 명칭을 생성합니다.")
#     feature_names = [f"feat_{i}" for i in range(data_test.edge_attr.shape[1])]

# def get_permutation_importance_gnn(model, loader, device, target_attr, base_ap, f_names):
#     import copy
#     importance_results = []

#     # 텐서를 직접 복사해서 조작 (GNN 특성 반영)
#     orig_edge_attr = loader.data.edge_attr.clone()

#     for i, col in enumerate(f_names):
#         # 1. 특정 피처 치환 (Shuffle)
#         perm = torch.randperm(orig_edge_attr.size(0))
#         shuffled_attr = orig_edge_attr.clone()
#         shuffled_attr[:, i] = orig_edge_attr[perm, i]

#         # 2. 로더의 데이터를 잠시 교체
#         loader.data.edge_attr = shuffled_attr

#         # 3. 성능 측정
#         y_perm, p_perm = evaluate(model, loader, device, target_attr)
#         p_perm = np.nan_to_num(p_perm, nan=0.0)
#         _, perm_ap = safe_auc(y_perm, p_perm)

#         # 4. 하락폭 계산
#         drop = base_ap - perm_ap
#         importance_results.append((col, max(0, drop)))

#         # 5. 원복
#         loader.data.edge_attr = orig_edge_attr
#         print(f"  - [{i+1}/{len(f_names)}] {col}: {drop:.4f}")

#     return sorted(importance_results, key=lambda x: x[1], reverse=True)

# # 실행
# gnn_imp_rows = get_permutation_importance_gnn(model, test_loader, device, test_target_edge_attr, t_ap, feature_names)

# # WandB 로깅
# imp_df = pd.DataFrame(gnn_imp_rows, columns=["feature", "importance"])
# wandb.log({"gnn_permutation_importance_table": wandb.Table(dataframe=imp_df)})

# top_features_for_analysis = [x[0] for x in gnn_imp_rows[:10]]

# # =======================================================
# # ★ LAS-GNN 상세 분석 및 비교 지표 추출 섹션
# # =======================================================

# # 1. 분석용 임계치(Threshold) 결정 (Val Recall 0.9 기준)
# def threshold_at_recall(y_true, y_score, desired_recall=0.90):
#     prec, rec, thr = precision_recall_curve(y_true, y_score)
#     if len(thr) == 0: return 0.5
#     closest_i = np.argmin(np.abs(rec[1:] - desired_recall))
#     return float(thr[closest_i])

# chosen_thr = threshold_at_recall(y_v, p_v, 0.90)

# # 2. 분석에 사용할 상위 피처 (GNN은 Feature Importance가 없으므로 임의 지정하거나 생략 가능)
# # 비교를 위해 원본 데이터셋의 주요 컬럼 몇 개를 지정합니다.
# # top_features_for_analysis = []

# print(f"\n[Analysis] Chosen Threshold: {chosen_thr:.4f}")
# print(f"[Analysis] Using Top Features: {top_features_for_analysis}")

# # 만약 test_df가 함수 안에 갇혀 있어서 못 불러오는 경우를 위한 방어 코드
# if 'test_df' not in locals() and 'test_df' not in globals():
#     print("[ERROR] test_df가 정의되지 않았습니다. 데이터를 로드한 변수명을 확인하세요.")
#     # 보통 test_df = pd.read_csv(...) 형태로 되어있어야 합니다.

# # --- (1) 일별/운영 KPI 로깅 ---
# log_per_day_kpi_to_wandb("test", df_test, y_t, p_t, k_list=(30, 50, 100, 200))

# # --- (2) 계좌 단위(Account-level) 성능 분석 ---
# log_per_day_account_kpi_to_wandb("test", df_test, y_t, p_t, k_list=(30, 50, 100, 200))

# # --- (3) 패턴별 검출 성능 (Find-K 기준) ---
# t_sum_3000, t_det_3000 = get_pattern_analysis_tables_find_k(df_test, p_t, 3000)
# wandb.log({
#     "GNN_Analysis/Pattern_Summary_Find3000": t_sum_3000,
#     "GNN_Analysis/Pattern_Detail_Find3000": t_det_3000
# })

# # --- (4) 정탐/오탐/미탐 상세 분석 (Find-3000 기준) ---
# log_find_k_error_analysis(
#     split_name="test_gnn",
#     df=df_test,
#     y_true=y_t,
#     y_score=p_t,
#     k_pos_target=3000,
#     top_features=top_features_for_analysis
# )

# # --- (5) Confusion Matrix 및 운영 지표 ---
# y_t_pred = (p_t >= chosen_thr).astype(int)
# cm_t = confusion_matrix(y_t, y_t_pred)
# fig_cm = plt.figure(figsize=(6,4))
# sns.heatmap(cm_t, annot=True, fmt="d", cmap="Greens") # GNN은 초록색으로 구분
# plt.title(f"GNN Test CM @thr={chosen_thr:.4f}")
# wandb.log({"test_gnn_confusion_matrix": wandb.Image(fig_cm)})
# plt.close(fig_cm)

# # --- (6) Score Bin Distribution ---
# dist_test = score_bin_distribution(y_t, p_t, n_bins=10, score_scale=1000)
# wandb.log({"test_gnn_score_bin_table": wandb.Table(dataframe=dist_test)})

print("✅ 모든 LAS-GNN 비교 지표 로깅이 완료되었습니다.")

In [43]:

# =======================================================
# ★ GNN Permutation Importance 계산 (수동 지정 대체)
# =======================================================
print("\n📊 Calculating Permutation Importance for GNN...")

# 0. feature_names가 없다면 강제로 생성 (오류 방지)
# 만약 전처리 단계에서 저장한 리스트가 있다면 그것을 쓰세요.
if 'feature_names' not in locals() and 'feature_names' not in globals():
    # 예: "Amount", "Timestamp" 등 텐서의 열 순서와 일치해야 함
    print("[WARN] feature_names를 찾을 수 없어 기본 명칭을 생성합니다.")
    feature_names = [f"feat_{i}" for i in range(data_test.edge_attr.shape[1])]

def get_permutation_importance_gnn(model, loader, device, target_attr, base_ap, f_names):
    import copy
    importance_results = []

    # 텐서를 직접 복사해서 조작 (GNN 특성 반영)
    orig_edge_attr = loader.data.edge_attr.clone()

    for i, col in enumerate(f_names):
        # 1. 특정 피처 치환 (Shuffle)
        perm = torch.randperm(orig_edge_attr.size(0))
        shuffled_attr = orig_edge_attr.clone()
        shuffled_attr[:, i] = orig_edge_attr[perm, i]

        # 2. 로더의 데이터를 잠시 교체
        loader.data.edge_attr = shuffled_attr

        # 3. 성능 측정
        y_perm, p_perm = evaluate(model, loader, device, target_attr)
        p_perm = np.nan_to_num(p_perm, nan=0.0)
        _, perm_ap = safe_auc(y_perm, p_perm)

        # 4. 하락폭 계산
        drop = base_ap - perm_ap
        importance_results.append((col, max(0, drop)))

        # 5. 원복
        loader.data.edge_attr = orig_edge_attr
        print(f"  - [{i+1}/{len(f_names)}] {col}: {drop:.4f}")

    return sorted(importance_results, key=lambda x: x[1], reverse=True)

# 실행
gnn_imp_rows = get_permutation_importance_gnn(model, test_loader, device, test_target_edge_attr, t_ap, feature_names)

# WandB 로깅
imp_df = pd.DataFrame(gnn_imp_rows, columns=["feature", "importance"])
wandb.log({"gnn_permutation_importance_table": wandb.Table(dataframe=imp_df)})

top_features_for_analysis = [x[0] for x in gnn_imp_rows[:10]]


📊 Calculating Permutation Importance for GNN...
  - [1/34] feat_0: 0.0194
  - [2/34] feat_1: -0.0019
  - [3/34] feat_2: 0.0023
  - [4/34] feat_3: 0.0120
  - [5/34] feat_4: 0.0099
  - [6/34] feat_5: 0.0016
  - [7/34] feat_6: -0.0037
  - [8/34] feat_7: -0.0037
  - [9/34] feat_8: 0.0001
  - [10/34] feat_9: 0.0116
  - [11/34] feat_10: 0.0440
  - [12/34] feat_11: -0.0028
  - [13/34] feat_12: 0.0100
  - [14/34] feat_13: 0.0185
  - [15/34] feat_14: -0.0016
  - [16/34] feat_15: 0.0161
  - [17/34] feat_16: -0.0003
  - [18/34] feat_17: -0.0003
  - [19/34] feat_18: 0.0269
  - [20/34] feat_19: 0.0097
  - [21/34] feat_20: 0.4208
  - [22/34] feat_21: 0.0035
  - [23/34] feat_22: -0.0045
  - [24/34] feat_23: 0.0022
  - [25/34] feat_24: 0.0026
  - [26/34] feat_25: -0.0030
  - [27/34] feat_26: 0.0622
  - [28/34] feat_27: 0.0638
  - [29/34] feat_28: 0.0209
  - [30/34] feat_29: 0.0549
  - [31/34] feat_30: 0.0367
  - [32/34] feat_31: 0.0695
  - [33/34] feat_32: 0.0571
  - [34/34] feat_33: 0.0052


Error: You must call wandb.init() before wandb.log()

In [46]:
top_features_for_analysis = [x[0] for x in gnn_imp_rows[:10]]

# --- (4) 정탐/오탐/미탐 상세 분석 (Find-3000 기준) ---
log_find_k_error_analysis(
    split_name="test_gnn",
    df=df_test,
    y_true=y_t,
    y_score=p_t,
    k_pos_target=3000,
    top_features=top_features_for_analysis
)


############################## [ test_gnn Error Analysis ] ##############################
Target K: 3000 | Required Workload (N): 3502
Precision: 0.8567 | Recall: 0.2820 | F1: 0.4243

[1. Daily Error Trend (test_gnn)]
    ts_day  FN (미탐)  FP (오탐)      TN  TP (정탐)
2022-09-14     1405       61 1417681      482
2022-09-15     1770       78 1928113      483
2022-09-16     1886      214 3019268      530
2022-09-17      742       32     789      457
2022-09-18      562       46     629      354
2022-09-19      475       27     473      220
2022-09-20      327       19     344      165
2022-09-21      220       14     236      116
2022-09-22      111        6     119       73
2022-09-23       47        4      46       37
2022-09-24       34        0      39       31
2022-09-25       34        0      43       25
2022-09-26       18        0      22       19
2022-09-27        9        0      11        5
2022-09-28        0        1       3        3

[2. Feature Statistics by Category]
category

{'daily_trend': error_cat     ts_day  FN (미탐)  FP (오탐)       TN  TP (정탐)
 0         2022-09-14     1405       61  1417681      482
 1         2022-09-15     1770       78  1928113      483
 2         2022-09-16     1886      214  3019268      530
 3         2022-09-17      742       32      789      457
 4         2022-09-18      562       46      629      354
 5         2022-09-19      475       27      473      220
 6         2022-09-20      327       19      344      165
 7         2022-09-21      220       14      236      116
 8         2022-09-22      111        6      119       73
 9         2022-09-23       47        4       46       37
 10        2022-09-24       34        0       39       31
 11        2022-09-25       34        0       43       25
 12        2022-09-26       18        0       22       19
 13        2022-09-27        9        0       11        5
 14        2022-09-28        0        1        3        3,
 'feat_stats':   category  count
 0  TP (정탐)   3000
 1  

In [ ]:

# =======================================================
# ★ LAS-GNN 상세 분석 및 비교 지표 추출 섹션
# =======================================================

# 1. 분석용 임계치(Threshold) 결정 (Val Recall 0.9 기준)
def threshold_at_recall(y_true, y_score, desired_recall=0.90):
    prec, rec, thr = precision_recall_curve(y_true, y_score)
    if len(thr) == 0: return 0.5
    closest_i = np.argmin(np.abs(rec[1:] - desired_recall))
    return float(thr[closest_i])

chosen_thr = threshold_at_recall(y_v, p_v, 0.90)

# 2. 분석에 사용할 상위 피처 (GNN은 Feature Importance가 없으므로 임의 지정하거나 생략 가능)
# 비교를 위해 원본 데이터셋의 주요 컬럼 몇 개를 지정합니다.
# top_features_for_analysis = []

print(f"\n[Analysis] Chosen Threshold: {chosen_thr:.4f}")
print(f"[Analysis] Using Top Features: {top_features_for_analysis}")

# 만약 test_df가 함수 안에 갇혀 있어서 못 불러오는 경우를 위한 방어 코드
if 'test_df' not in locals() and 'test_df' not in globals():
    print("[ERROR] test_df가 정의되지 않았습니다. 데이터를 로드한 변수명을 확인하세요.")
    # 보통 test_df = pd.read_csv(...) 형태로 되어있어야 합니다.

# --- (1) 일별/운영 KPI 로깅 ---
log_per_day_kpi_to_wandb("test", df_test, y_t, p_t, k_list=(30, 50, 100, 200))

# --- (2) 계좌 단위(Account-level) 성능 분석 ---
log_per_day_account_kpi_to_wandb("test", df_test, y_t, p_t, k_list=(30, 50, 100, 200))

# --- (3) 패턴별 검출 성능 (Find-K 기준) ---
t_sum_3000, t_det_3000 = get_pattern_analysis_tables_find_k(df_test, p_t, 3000)
wandb.log({
    "GNN_Analysis/Pattern_Summary_Find3000": t_sum_3000,
    "GNN_Analysis/Pattern_Detail_Find3000": t_det_3000
})

# --- (4) 정탐/오탐/미탐 상세 분석 (Find-3000 기준) ---
log_find_k_error_analysis(
    split_name="test_gnn",
    df=df_test,
    y_true=y_t,
    y_score=p_t,
    k_pos_target=3000,
    top_features=top_features_for_analysis
)

# --- (5) Confusion Matrix 및 운영 지표 ---
y_t_pred = (p_t >= chosen_thr).astype(int)
cm_t = confusion_matrix(y_t, y_t_pred)
fig_cm = plt.figure(figsize=(6,4))
sns.heatmap(cm_t, annot=True, fmt="d", cmap="Greens") # GNN은 초록색으로 구분
plt.title(f"GNN Test CM @thr={chosen_thr:.4f}")
wandb.log({"test_gnn_confusion_matrix": wandb.Image(fig_cm)})
plt.close(fig_cm)

# --- (6) Score Bin Distribution ---
dist_test = score_bin_distribution(y_t, p_t, n_bins=10, score_scale=1000)
wandb.log({"test_gnn_score_bin_table": wandb.Table(dataframe=dist_test)})

print("✅ 모든 LAS-GNN 비교 지표 로깅이 완료되었습니다.")

In [40]:
# --- (3) 패턴별 검출 성능 (Find-K 기준) ---
t_sum_3000, t_det_3000 = get_pattern_analysis_tables_find_k(df_test, p_t, 3000)
wandb.log({
    "GNN_Analysis/Pattern_Summary_Find3000": t_sum_3000,
    "GNN_Analysis/Pattern_Detail_Find3000": t_det_3000
})


==================== Find-K Pattern Analysis (Target K: 3000) ====================
Required Workload (N): 3502

[1. Pattern Type Summary]
  Pattern_Type  Total_Actual  Detected_Count  Recall_Pct
          NONE          2496               5        0.20
        FAN-IN           849             240       28.27
        RANDOM           541             165       30.50
         STACK          1429             448       31.35
     BIPARTITE           501             172       34.33
         CYCLE           723             267       36.93
SCATTER-GATHER          1295             495       38.22
GATHER-SCATTER          2182             939       43.03
       FAN-OUT           624             269       43.11

[2. Pattern Detail Analysis]
  Pattern_Type        Pattern_Detail  Total_Actual  Detected_Count  Recall_Pct
         CYCLE           Max 11 hops            70              17       24.29
         CYCLE           Max 10 hops            45              13       28.89
         CYCLE          

Error: You must call wandb.init() before wandb.log()

In [47]:
import pyarrow as pa
import pyarrow.parquet as pq

@torch.no_grad()
def extract_and_save_parquet(model, loader, device, target_edge_attr_full,
                              split_name, src_cols, dst_cols, out_path):

    model.eval()
    writer = None
    all_scores, all_labels = [], []

    for batch in tqdm(loader, desc=f"Extracting {split_name}"):
        batch = batch.to(device)
        tef = target_edge_attr_full[batch.input_id.cpu()].to(device)

        h = model.encode_nodes(
            batch.n_id, batch.edge_index, batch.edge_attr,
            getattr(batch, 'x', None)
        )
        src_h = h[batch.edge_label_index[0]]
        dst_h = h[batch.edge_label_index[1]]

        logit = model.classifier(torch.cat([src_h, dst_h, tef], dim=1)).squeeze(-1)
        pred = torch.sigmoid(logit)

        # numpy 변환 (이 배치만)
        src_np = src_h.cpu().numpy()
        dst_np = dst_h.cpu().numpy()
        pred_np = pred.cpu().numpy()

        # 바로 parquet에 기록
        data = {}
        for i in range(src_np.shape[1]):
            data[src_cols[i]] = src_np[:, i]
            data[dst_cols[i]] = dst_np[:, i]
        data["gnn_score"] = pred_np

        table = pa.Table.from_pydict(data)
        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema)
        writer.write_table(table)

        # AUPRC 계산용 (score만)
        all_labels.append(batch.edge_label.cpu().numpy())
        all_scores.append(pred_np)

    if writer:
        writer.close()

    y = np.concatenate(all_labels)
    p = np.concatenate(all_scores)
    auprc = average_precision_score(y, p)
    print(f"  ✅ Saved {out_path} | AUPRC={auprc:.4f} | rows={len(y):,}")
    return auprc


In [121]:
import gc

torch.cuda.empty_cache(); gc.collect()
print("🧹 Cache cleared!")

hidden_dim = 256
src_cols = [f"gnn_src_{i}" for i in range(hidden_dim)]
dst_cols = [f"gnn_dst_{i}" for i in range(hidden_dim)]

# ── Train ──
train_infer_loader = LinkNeighborLoader(
    data_train, num_neighbors=num_neighbors, batch_size=batch_size,
    edge_label_index=data_train.edge_index, edge_label=data_train.y,
    shuffle=False, num_workers=4)

extract_and_save_parquet(model, train_infer_loader, device, edge_attr_train,
                          "train", src_cols, dst_cols, "gnn_emb_train.parquet")
del train_infer_loader; torch.cuda.empty_cache(); gc.collect()

# ── Val ──
extract_and_save_parquet(model, val_loader, device, val_target_edge_attr,
                          "val", src_cols, dst_cols, "gnn_emb_val.parquet")
torch.cuda.empty_cache(); gc.collect()

# ── Test ──
extract_and_save_parquet(model, test_loader, device, test_target_edge_attr,
                          "test", src_cols, dst_cols, "gnn_emb_test.parquet")

print("\n✅ All done!")
print("   - gnn_emb_train.parquet")
print("   - gnn_emb_val.parquet")
print("   - gnn_emb_test.parquet")

🧹 Cache cleared!


Extracting train: 100%|██████████| 2337/2337 [08:02<00:00,  4.84it/s]


  ⬆️ Copying train to Drive...
  ✅ Done! gnn_emb_train.parquet | AUPRC=0.5130 | rows=19,139,731


Extracting val: 100%|██████████| 779/779 [03:23<00:00,  3.82it/s]


  ⬆️ Copying val to Drive...
  ✅ Done! gnn_emb_val.parquet | AUPRC=0.4934 | rows=6,379,980


Extracting test: 100%|██████████| 779/779 [03:43<00:00,  3.49it/s]


  ⬆️ Copying test to Drive...
  ✅ Done! gnn_emb_test.parquet | AUPRC=0.5091 | rows=6,378,958

✅ All done!
   - gnn_emb_train.parquet
   - gnn_emb_val.parquet
   - gnn_emb_test.parquet


In [51]:
import shutil
import os
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm
import torch
import numpy as np
from sklearn.metrics import average_precision_score

@torch.no_grad()
def extract_and_save_parquet(model, loader, device, target_edge_attr_full,
                              split_name, src_cols, dst_cols, final_drive_path):

    model.eval()
    # 1. 일단 코랩 로컬(속도가 빠른 경로)에 임시 저장
    temp_local_path = f"temp_{split_name}.parquet"
    writer = None
    all_scores, all_labels = [], []

    try:
        for batch in tqdm(loader, desc=f"Extracting {split_name}"):
            batch = batch.to(device)
            # input_id가 유효한지 확인 필요 (LinkNeighborLoader 설정에 따라 다름)
            tef = target_edge_attr_full[batch.input_id.cpu()].to(device)

            h = model.encode_nodes(
                batch.n_id, batch.edge_index, batch.edge_attr,
                getattr(batch, 'x', None)
            )
            src_h = h[batch.edge_label_index[0]]
            dst_h = h[batch.edge_label_index[1]]

            # Classifier & Sigmoid
            logit = model.classifier(torch.cat([src_h, dst_h, tef], dim=1)).squeeze(-1)
            pred = torch.sigmoid(logit)

            # CPU 변환 및 저장용 데이터 구성
            src_np = src_h.cpu().numpy()
            dst_np = dst_h.cpu().numpy()
            pred_np = pred.cpu().numpy()

            # 효율적인 Table 생성 (딕셔너리 컴프리헨션)
            data = {src_cols[i]: src_np[:, i] for i in range(src_np.shape[1])}
            data.update({dst_cols[i]: dst_np[:, i] for i in range(dst_np.shape[1])})
            data["gnn_score"] = pred_np

            table = pa.Table.from_pydict(data)
            if writer is None:
                writer = pq.ParquetWriter(temp_local_path, table.schema)
            writer.write_table(table)

            all_labels.append(batch.edge_label.cpu().numpy())
            all_scores.append(pred_np)

        if writer:
            writer.close()

        # 2. 작업 완료 후 구글 드라이브로 복사
        print(f"  ⬆️ Copying {split_name} to Drive...")
        shutil.copy(temp_local_path, final_drive_path)
        os.remove(temp_local_path) # 로컬 임시 파일 삭제

        y = np.concatenate(all_labels)
        p = np.concatenate(all_scores)
        auprc = average_precision_score(y, p)
        print(f"  ✅ Done! {final_drive_path} | AUPRC={auprc:.4f} | rows={len(y):,}")
        return auprc

    except Exception as e:
        if writer: writer.close()
        print(f"  ❌ Error during {split_name}: {e}")
        return None

# 실행 부분은 동일하되, drive.flush_and_unmount() 권장
# ... (마운트 코드 생략) ...

# ── Train ──
# (기존 코드와 동일하게 호출하되, 내부에서 로컬->드라이브 복사 수행)


In [122]:
def train_eval_sweep():
    """
    W&B agent가 호출하는 sweep train function.
    - run 안에서 transformer fit은 train만
    - val/test는 transform만 (누수 방지)
    - scale_pos_weight sweep
    - val 기준으로 sweep 최적화 metric 로깅
    - PR curve / CM / Top-K는 val/test 각각 로깅
    - XGB feature importance(gain) 로깅 (변환 후 feature name)
    """
    run = wandb.init()
    config = wandb.config
    set_config(transform_output="pandas")

    # ---------------------------------------------------------
    # (0) GNN 임베딩 로드 및 병합 (이 부분이 추가되었습니다)
    # ---------------------------------------------------------
    # def merge_gnn_embeddings(main_df, split_name):
    #     emb_path = f"/content/drive/MyDrive/GNN_Embeddings/gnn_emb_{split_name}.parquet"
    #     if not os.path.exists(emb_path):
    #         print(f"[ERROR] Embedding file not found: {emb_path}")
    #         return main_df

    #     # 임베딩 데이터 로드 (node_id, emb_0, emb_1, ... 형태 가정)
    #     emb_pl = pl.read_parquet(emb_path)

    #     # 병합을 위해 메인 DF를 Polars로 변환
    #     main_pl = pl.from_pandas(main_df)

    #     # 1. Sender 임베딩 결합
    #     main_pl = main_pl.join(
    #         emb_pl.rename({c: f"s_{c}" for c in emb_pl.columns if c != "node_id"}),
    #         left_on="sender_node",
    #         right_on="node_id",
    #         how="left"
    #     )

    #     # 2. Receiver 임베딩 결합
    #     main_pl = main_pl.join(
    #         emb_pl.rename({c: f"r_{c}" for c in emb_pl.columns if c != "node_id"}),
    #         left_on="receiver_node",
    #         right_on="node_id",
    #         how="left"
    #     )

    #     return main_pl.to_pandas()

    # print("Merging GNN Embeddings...")
    # # train_df, val_df, test_df는 외부 스코프에서 정의된 것을 사용
    # X_train_merged = merge_gnn_embeddings(train_df, "train")
    # X_val_merged   = merge_gnn_embeddings(val_df,   "val")
    # X_test_merged  = merge_gnn_embeddings(test_df,  "test")

    def merge_gnn_embeddings(main_df, split_name):
        # 1. 파일 경로 수정 (이미지 상으로 /content/ 바로 아래에 파일들이 보입니다)
        # 만약 에러가 계속 나면 아래 경로를 '/content/gnn_emb_{split_name}.parquet'로 바꿔보세요.
        emb_path = f"gnn_emb_{split_name}.parquet"

        if not os.path.exists(emb_path):
            # 경로 재확인용 로직
            alternative_path = f"/content/{emb_path}"
            if os.path.exists(alternative_path):
                emb_path = alternative_path
            else:
                print(f"[ERROR] Embedding file not found: {emb_path}")
                return main_df

        # 2. 임베딩 데이터 로드
        emb_pl = pl.read_parquet(emb_path)

        # [중요] 에러 로그를 분석한 결과, 이미 임베딩 파일에
        # 'node_id'가 없고 다른 컬럼들이 가득 차 있습니다.
        # 만약 이 파일이 main_df(거래 데이터)와 1:1로 행 순서가 맞는 'Feature' 파일이라면
        # join이 아니라 가로로 붙여야(concat) 할 수도 있습니다.

        # 하지만 안전하게 'sender_node' 혹은 'receiver_node'와 매칭되는
        # ID 컬럼이 파일 안에 있는지 먼저 찾아야 합니다.

        # 에러 메시지의 'valid columns' 목록에 ID로 쓸만한 게 없다면
        # 이 임베딩 파일은 이미 거래 데이터와 행 순서가 일치하게 생성된 것입니다.
        # 이 경우 아래와 같이 처리합니다.

        print(f"Applying GNN features from {emb_path} (Row-wise concat)...")

        main_pl = pl.from_pandas(main_df)

        # 만약 행 수가 같다면 단순 수평 결합을 진행합니다.
        if len(main_pl) == len(emb_pl):
            # 중복 컬럼 피하기 위해 임베딩 파일의 컬럼명에 접두어 추가
            emb_pl = emb_pl.rename({c: f"gnn_{c}" for c in emb_pl.columns})
            return pl.concat([main_pl, emb_pl], how="horizontal").to_pandas()
        else:
            # 행 수가 다르다면, ID 컬럼을 다시 찾아야 합니다.
            # (파일을 생성한 곳에서 ID 컬럼명을 확인해야 합니다. 보통 'Account' 등)
            print("[WARN] Row counts mismatch! Check if ID column exists in embedding file.")
            return main_df

    # 함수 실행부 위에서 경로 확인을 한 번 더 해주세요.
    import os
    print("Current files:", os.listdir('/content/'))

    # =========================
    # (0) RAW -> X/y 분리
    # =========================
    LABEL_COL = "Is Laundering"

    # train/val/test는 "이미 time split 완료된 pandas df"라고 가정
    # (run 밖에서 만들어둔 걸 그대로 씀)
    X_train_raw = train_df.drop(columns=[LABEL_COL, "ts_day", "ts", "bucket_ts", "From Account", "To Account", "Pattern_Type", "Pattern_Detail", "sender_node", "receiver_node"]) # 계좌O 버전에서는 주석처리 "From Account", "To Account",
    y_train     = train_df[LABEL_COL].astype(int)

    X_val_raw   = val_df.drop(columns=[LABEL_COL, "ts_day", "ts", "bucket_ts", "From Account", "To Account", "Pattern_Type", "Pattern_Detail", "sender_node", "receiver_node"]) # 계좌O 버전에서는 주석처리 "From Account", "To Account",
    y_val       = val_df[LABEL_COL].astype(int)

    X_test_raw  = test_df.drop(columns=[LABEL_COL, "ts_day", "ts", "bucket_ts", "From Account", "To Account", "Pattern_Type", "Pattern_Detail", "sender_node", "receiver_node"]) # 계좌O 버전에서는 주석처리 "From Account", "To Account",
    y_test      = test_df[LABEL_COL].astype(int)

    # =========================
    # (0.5) Safety check: forbidden columns
    # =========================
    FORBIDDEN_COLS = {
        "From Account", # 계좌O 버전에서는 주석처리
        "To Account", # 계좌O 버전에서는 주석처리
        "sender_node",
        "receiver_node",
        "ts",
        "bucket_ts",
        "bucket_ts_str",
        "Timestamp",
    }

    for split_name, X in [
        ("train", X_train_raw),
        ("val",   X_val_raw),
        ("test",  X_test_raw),
    ]:
        leaked = FORBIDDEN_COLS.intersection(X.columns)
        assert len(leaked) == 0, (
            f"[LEAKAGE ERROR] {split_name} input contains forbidden columns: {leaked}"
        )

    # -------------------------
    # 0) 컬럼 그룹 정의
    # -------------------------
    NATIVE_CAT_COLS = [
        "dow_cat",
        "Receiving Currency",
        "Payment Currency",
        "Payment Format",
        "timeofday_bucket",
        "Sender_Bank_Branch_ID",
        "Receiver_Bank_Branch_ID",
    ]

    HIGH_CARD_COLS = [
        "Sender_Entity",
        "Sender_Bank_Name",
        "Receiver_Entity",
        "Receiver_Bank_Name",
        # "From Account", # 계좌O에서는 주석 해제
        # "To Account",  # 계좌O에서는 주석 해제
        "From Country",
        "To Country",
        "From Bank",
        "To Bank",
    ]

    # -------------------------
    # 1) Top-K + RARE (train-only) 사전 만들기 + 적용
    #    - 누수 방지: fit은 train에서만
    # -------------------------
    def fit_topk_maps(df_train: pd.DataFrame, cols, top_k=2000, min_count=None):
        """
        cols 각각에 대해:
        - top_k: 가장 많이 나온 카테고리 top_k만 유지
        - min_count: (선택) 빈도 min_count 미만은 RARE로
        반환: {col: set(allowed_categories)}
        """
        maps = {}
        for c in cols:
            if c not in df_train.columns:
                continue
            s = df_train[c].astype("string").fillna("__MISSING__")
            vc = s.value_counts(dropna=False)

            if min_count is not None:
                allowed = set(vc[vc >= min_count].index.astype(str))
            else:
                allowed = set(vc.head(top_k).index.astype(str))

            # MISSING은 항상 허용(해석 가능 + 안정)
            allowed.add("__MISSING__")
            maps[c] = allowed
        return maps

    def apply_topk_maps(df: pd.DataFrame, maps, rare_token="__RARE__"):
        df = df.copy()
        for c, allowed in maps.items():
            if c not in df.columns:
                continue
            s = df[c].astype("string").fillna("__MISSING__")
            df[c] = np.where(s.isin(list(allowed)), s, rare_token)
        return df

    # -------------------------
    # 2) category dtype 강제 (XGBoost native categorical용)
    # -------------------------
    def force_category(df: pd.DataFrame, cols):
        df = df.copy()
        for c in cols:
            if c in df.columns:
                df[c] = df[c].astype("category")
        return df

    # ---- (A) train 기준 top-k 사전 fit
    topk_maps = fit_topk_maps(X_train_raw, HIGH_CARD_COLS, top_k=2000, min_count=None)

    # ---- (B) train/val/test에 동일하게 적용 (누수 없음)
    X_train_raw2 = apply_topk_maps(X_train_raw, topk_maps)
    X_val_raw2   = apply_topk_maps(X_val_raw,   topk_maps)
    X_test_raw2  = apply_topk_maps(X_test_raw,  topk_maps)

    # ---- (C) native-cat + high-card(정규화된) 모두 category로 고정
    CAT_COLS_FINAL = [c for c in (NATIVE_CAT_COLS + HIGH_CARD_COLS) if c in X_train_raw2.columns]

    X_train_raw2 = force_category(X_train_raw2, CAT_COLS_FINAL)
    X_val_raw2   = force_category(X_val_raw2,   CAT_COLS_FINAL)
    X_test_raw2  = force_category(X_test_raw2,  CAT_COLS_FINAL)

    # -------------------------
    # bool 컬럼은 numeric으로 캐스팅 (median impute 안전)
    #    (너 로그에 떨어져나간 애들 대부분 bool일 가능성이 큼)
    # -------------------------
    def cast_bool_to_int(df):
        df = df.copy()
        bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
        for c in bool_cols:
            df[c] = df[c].astype("int8")
        return df

    X_train_raw2 = cast_bool_to_int(X_train_raw2)
    X_val_raw2   = cast_bool_to_int(X_val_raw2)
    X_test_raw2  = cast_bool_to_int(X_test_raw2)

    # -------------------------
    # 4) 전처리기: numeric만 impute, categorical은 passthrough로 "진짜 category" 유지
    #    + 결과를 DataFrame으로 받게 됨 (set_config 덕분)
    # -------------------------
    all_cols = X_train_raw2.columns.tolist()
    num_cols = [c for c in all_cols if c not in CAT_COLS_FINAL]
    cat_cols = [c for c in CAT_COLS_FINAL if c in all_cols]

    # 3) 혹시라도 num/cat 둘 다 아닌 "문자열 잔여 컬럼"이 있다면
    #    -> 지금은 안전하게 drop (원하면 나중에 별도 처리)
    rest_cols = [c for c in X_train_raw2.columns if (c not in num_cols and c not in cat_cols)]
    if len(rest_cols) > 0:
        print("[WARN] Dropping non-numeric non-category cols:", rest_cols)
        X_train_raw2 = X_train_raw2.drop(columns=rest_cols)
        X_val_raw2   = X_val_raw2.drop(columns=rest_cols)
        X_test_raw2  = X_test_raw2.drop(columns=rest_cols)

    # 4) transformer 구성 (num만 impute, cat은 passthrough)
    num_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    transformer = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
        ],
        remainder="passthrough",   # cat_cols는 그대로 뒤에 붙음
        verbose_feature_names_out=False,
    )

    X_train = transformer.fit_transform(X_train_raw2)
    X_val   = transformer.transform(X_val_raw2)
    X_test  = transformer.transform(X_test_raw2)

    # ✅ passthrough로 온 cat 컬럼이 혹시 object로 바뀌었으면 다시 category로 고정
    # (환경/버전에 따라 가끔 필요)
    for c in CAT_COLS_FINAL:
        if c in X_train.columns:
            X_train[c] = X_train[c].astype("category")
            X_val[c]   = X_val[c].astype("category")
            X_test[c]  = X_test[c].astype("category")

    # feature names
    feature_names = X_train.columns.tolist()

    # =========================
    # (DEBUG) categorical dtype 살아있는지 확인
    # =========================
    print("=== Raw2 categorical dtypes ===")
    print(X_train_raw2[cat_cols].dtypes)

    print("\n=== Transformed X_train info ===")
    print("type(X_train):", type(X_train))
    print("X_train dtype:", getattr(X_train, "dtype", None))

    # =========================
    # (A) 모델 생성 (sweep 파라미터)
    # =========================
    xgb_model = XGBClassifier(
        n_estimators=config.n_estimators,
        max_depth=config.max_depth,
        learning_rate=config.learning_rate,
        subsample=config.subsample,
        colsample_bytree=config.colsample_bytree,
        reg_lambda=config.reg_lambda,
        reg_alpha=config.reg_alpha,
        min_child_weight=config.min_child_weight,
        gamma=config.gamma,
        scale_pos_weight=config.scale_pos_weight,   # ✅ sweep
        tree_method="hist",
        enable_categorical=True,
        objective="binary:logistic",  # 기본
        eval_metric="aucpr",           # 학습 모니터링용
        random_state=42,
        n_jobs=-1
    )

    # =========================
    # (B) 학습
    # =========================
    xgb_model.fit(X_train, y_train)

    # =========================
    # (C) 평가 (val/test 둘 다)
    # =========================
    def safe_auc_metrics(y_true, y_score):
        if len(np.unique(y_true)) <= 1:
            return np.nan, np.nan
        return roc_auc_score(y_true, y_score), average_precision_score(y_true, y_score)

    def topk_metrics(y_true, y_score, k):
        k = min(k, len(y_true))
        idx = np.argsort(-y_score)[:k]
        y_pred = np.zeros_like(y_true)
        y_pred[idx] = 1
        p = precision_score(y_true, y_pred, zero_division=0)
        r = recall_score(y_true, y_pred, zero_division=0)
        f = f1_score(y_true, y_pred, zero_division=0)
        return p, r, f

    def threshold_at_recall(y_true, y_score, desired_recall=0.90):
        prec, rec, thr = precision_recall_curve(y_true, y_score)
        if len(thr) == 0:
            return 0.5, prec, rec, thr
        closest_i = np.argmin(np.abs(rec[1:] - desired_recall))
        return float(thr[closest_i]), prec, rec, thr

    # -------------------------
    # (C-1) val / test 확률 예측
    # -------------------------
    prob_val  = xgb_model.predict_proba(X_val)[:, 1]
    prob_test = xgb_model.predict_proba(X_test)[:, 1]

    val_roc_auc,  val_auprc  = safe_auc_metrics(y_val, prob_val)
    test_roc_auc, test_auprc = safe_auc_metrics(y_test, prob_test)

    # ✅ sweep 최적화 metric은 val 기준으로 통일
    wandb.log({
        "val_roc_auc":  val_roc_auc,
        "val_auprc":    val_auprc,
        "test_roc_auc": test_roc_auc,
        "test_auprc":   test_auprc,
        "n_train": len(y_train),
        "pos_train": int(np.sum(y_train)),
        "pos_val": int(np.sum(y_val)),
        "pos_test": int(np.sum(y_test)),
    })

    # -------------------------
    # (C-2) PR Curve 로깅 (val/test 각각)
    # -------------------------
    # PR curve 로깅 (val/test)
    prec_v, rec_v, _ = precision_recall_curve(y_val, prob_val)
    fig_pr_v = plt.figure()
    plt.plot(rec_v, prec_v)
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"VAL PR Curve (AUPRC={val_auprc:.4f})")
    wandb.log({"val_pr_curve": wandb.Image(fig_pr_v)})
    plt.close(fig_pr_v)

    prec_t, rec_t, _ = precision_recall_curve(y_test, prob_test)
    fig_pr_t = plt.figure()
    plt.plot(rec_t, prec_t)
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"TEST PR Curve (AUPRC={test_auprc:.4f})")
    wandb.log({"test_pr_curve": wandb.Image(fig_pr_t)})
    plt.close(fig_pr_t)

    # -------------------------
    # (C-3) Top-k
    # -------------------------
    # "K TP 찾기 위한 조사량" 로깅 (val/test 각각)
    for k_pos in [450, 750, 1500, 3000]:
        N_v, p_v, r_v, f_v, found_v = workload_to_find_k_positives(y_val.values, prob_val, k_pos)
        N_t, p_t, r_t, f_t, found_t = workload_to_find_k_positives(y_test.values, prob_test, k_pos)

        wandb.log({
            # VAL
            f"val_find_{k_pos}_N_required": N_v,
            f"val_find_{k_pos}_precision": p_v,
            f"val_find_{k_pos}_recall":    r_v,
            f"val_find_{k_pos}_f1":        f_v,
            f"val_find_{k_pos}_found_tp":  found_v,

            # TEST
            f"test_find_{k_pos}_N_required": N_t,
            f"test_find_{k_pos}_precision": p_t,
            f"test_find_{k_pos}_recall":    r_t,
            f"test_find_{k_pos}_f1":        f_t,
            f"test_find_{k_pos}_found_tp":  found_t,
        })

    # per-day KPI 로깅 (VAL/TEST 각각)
    # 전제: val_df/test_df에 ts_day 컬럼 존재
    log_per_day_kpi_to_wandb("val",  val_df,  y_val.values,  prob_val,  k_list=(30,50,100,200))
    log_per_day_kpi_to_wandb("test", test_df, y_test.values, prob_test, k_list=(30,50,100,200))

    days_test = test_df["ts_day"].nunique()
    topN = 30 * days_test  # 예: 450

    # res = topN_distinct_account_metrics(
    #     df_meta=test_df, y_true=y_test.values, y_score=prob_test,
    #     topN=topN, account_col="From Account"
    # )

    res = topN_distinct_account_metrics(
        df_meta=test_df, y_true=y_test.values, y_score=prob_test,
        topN=topN, account_cols=("From Account","To Account"),  # ✅
        score_agg="max"
    )

    wandb.log({
        "test_cum_topN": res["topN_trx"],
        "test_cum_distinct_accounts": res["distinct_accounts"],
        "test_cum_tp_accounts": res["tp_accounts"],
        "test_cum_fp_accounts": res["fp_accounts"],
        "test_cum_precision_accounts": res["precision_accounts"],
    })

    dist_val  = score_bin_distribution(y_val.values,  prob_val,  n_bins=10, score_scale=1000)
    dist_test = score_bin_distribution(y_test.values, prob_test, n_bins=10, score_scale=1000)
    wandb.log({
        "val_score_bin_table":  wandb.Table(dataframe=dist_val),
        "test_score_bin_table": wandb.Table(dataframe=dist_test),
    })

    # =========================
    # (C-3b) A count-level KPI (VAL/TEST)
    # =========================

    # 1) Account-level AUC/AUPRC (기간 전체, From+To 합산)
    val_acc = aggregate_to_account_level(val_df, y_val.values, prob_val,
                                         account_cols=("From Account","To Account"),
                                         score_agg="max")
    test_acc = aggregate_to_account_level(test_df, y_test.values, prob_test,
                                          account_cols=("From Account","To Account"),
                                          score_agg="max")

    val_acc_roc, val_acc_auprc = safe_auc_metrics(val_acc["label"].values, val_acc["score"].values)
    test_acc_roc, test_acc_auprc = safe_auc_metrics(test_acc["label"].values, test_acc["score"].values)

    wandb.log({
        "val_acc_roc_auc": val_acc_roc,
        "val_acc_auprc": val_acc_auprc,
        "test_acc_roc_auc": test_acc_roc,
        "test_acc_auprc": test_acc_auprc,
        "val_n_accounts": int(len(val_acc)),
        "test_n_accounts": int(len(test_acc)),
        "val_pos_accounts": int(val_acc["label"].sum()),
        "test_pos_accounts": int(test_acc["label"].sum()),
    })

    # 2) Account-level Find-K workload (기간 전체)
    for k_pos in [450, 750, 1500, 3000]:
        N_v, p_v, r_v, f_v, found_v = workload_to_find_k_positives(val_acc["label"].values, val_acc["score"].values, k_pos)
        N_t, p_t, r_t, f_t, found_t = workload_to_find_k_positives(test_acc["label"].values, test_acc["score"].values, k_pos)

        wandb.log({
            f"val_acc_find_{k_pos}_N_required": N_v,
            f"val_acc_find_{k_pos}_precision": p_v,
            f"val_acc_find_{k_pos}_recall":    r_v,
            f"val_acc_find_{k_pos}_f1":        f_v,
            f"val_acc_find_{k_pos}_found_tp":  found_v,

            f"test_acc_find_{k_pos}_N_required": N_t,
            f"test_acc_find_{k_pos}_precision": p_t,
            f"test_acc_find_{k_pos}_recall":    r_t,
            f"test_acc_find_{k_pos}_f1":        f_t,
            f"test_acc_find_{k_pos}_found_tp":  found_t,
        })

    # 3) per-day account KPI (하루 단위 운영 KPI를 계좌 기준으로)
    log_per_day_account_kpi_to_wandb("val",  val_df,  y_val.values,  prob_val,  k_list=(30,50,100,200))
    log_per_day_account_kpi_to_wandb("test", test_df, y_test.values, prob_test, k_list=(30,50,100,200))

    # -------------------------
    # (C-4) threshold@recall (val에서 threshold 선택 → test에 적용)
    # -------------------------
    desired_recall = 0.90
    chosen_thr, _, _, _ = threshold_at_recall(y_val, prob_val, desired_recall)

    # ✅ thr 기반 예측(한 번만)
    y_val_pred  = (prob_val  >= chosen_thr).astype(int)
    y_test_pred = (prob_test >= chosen_thr).astype(int)

    # ✅ CM (thr 기반)
    cm_val  = confusion_matrix(y_val,  y_val_pred)
    cm_test = confusion_matrix(y_test, y_test_pred)

    fig_cm_v = plt.figure(figsize=(6,4))
    sns.heatmap(cm_val, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(f"VAL CM @thr={chosen_thr:.4f}")
    wandb.log({"val_confusion_matrix": wandb.Image(fig_cm_v)})
    plt.close(fig_cm_v)

    fig_cm_t = plt.figure(figsize=(6,4))
    sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(f"TEST CM @thr={chosen_thr:.4f}")
    wandb.log({"test_confusion_matrix": wandb.Image(fig_cm_t)})
    plt.close(fig_cm_t)

    # =========================
    # (C-4-2) Precision / Recall / F1
    # =========================

    # (A) chosen_thr 기준 (운영)
    val_precision_thr  = precision_score(y_val,  y_val_pred,  zero_division=0)
    val_recall_thr     = recall_score(y_val,     y_val_pred,  zero_division=0)
    val_f1_thr         = f1_score(y_val,         y_val_pred,  zero_division=0)

    test_precision_thr = precision_score(y_test, y_test_pred, zero_division=0)
    test_recall_thr    = recall_score(y_test,    y_test_pred, zero_division=0)
    test_f1_thr        = f1_score(y_test,        y_test_pred, zero_division=0)

    # (B) 0.5 기준 (참고)
    thr_05 = 0.5
    y_val_pred_05  = (prob_val  >= thr_05).astype(int)
    y_test_pred_05 = (prob_test >= thr_05).astype(int)

    val_precision_05  = precision_score(y_val,  y_val_pred_05,  zero_division=0)
    val_recall_05     = recall_score(y_val,     y_val_pred_05,  zero_division=0)
    val_f1_05         = f1_score(y_val,         y_val_pred_05,  zero_division=0)

    test_precision_05 = precision_score(y_test, y_test_pred_05, zero_division=0)
    test_recall_05    = recall_score(y_test,    y_test_pred_05, zero_division=0)
    test_f1_05        = f1_score(y_test,        y_test_pred_05, zero_division=0)

    # ✅ 로그(한 번에)
    wandb.log({
        "desired_recall_for_thr": float(desired_recall),
        "chosen_threshold": float(chosen_thr),

        "val_precision_thr":  float(val_precision_thr),
        "val_recall_thr":     float(val_recall_thr),
        "val_f1_thr":         float(val_f1_thr),
        "test_precision_thr": float(test_precision_thr),
        "test_recall_thr":    float(test_recall_thr),
        "test_f1_thr":        float(test_f1_thr),

        "val_precision_thr_0.5":  float(val_precision_05),
        "val_recall_thr_0.5":     float(val_recall_05),
        "val_f1_thr_0.5":         float(val_f1_05),
        "test_precision_thr_0.5": float(test_precision_05),
        "test_recall_thr_0.5":    float(test_recall_05),
        "test_f1_thr_0.5":        float(test_f1_05),
    })

    # =========================
    # (C-5) Feature Importance (gain)
    # =========================
    booster = xgb_model.get_booster()
    gain_dict = booster.get_score(importance_type="gain")

    # (디버그) 지금 key가 뭐로 오는지 5개만 확인
    print("[DEBUG] gain_dict size:", len(gain_dict))
    print("[DEBUG] gain_dict keys sample:", list(gain_dict.keys())[:5])

    if len(gain_dict) == 0:
        print("[WARN] gain_dict is empty. (모델이 split을 거의 안 했거나, importance_type 변경 필요)")
    else:
        imp_rows = []
        for k, v in gain_dict.items():
            # 1) key가 f0,f1 형태면 feature_names로 매핑
            if isinstance(k, str) and k.startswith("f") and k[1:].isdigit():
                idx = int(k[1:])
                name = feature_names[idx] if idx < len(feature_names) else k
            else:
                # 2) key가 이미 컬럼명 형태면 그대로 사용
                name = str(k)
            imp_rows.append((name, float(v)))

        imp_rows.sort(key=lambda x: x[1], reverse=True)

        # (1) 전체 table 로깅
        table_all = wandb.Table(columns=["feature", "gain"])
        for name, gain in imp_rows:
            table_all.add_data(name, gain)
        wandb.log({"feature_importance_gain_table_all": table_all})

        # (2) bar는 top만
        topN_bar = int(getattr(config, "topn_importance", 50)) if hasattr(config, "topn_importance") else 50
        imp_top = imp_rows[:topN_bar]

        fig_imp = plt.figure(figsize=(8, 10))
        names = [x[0] for x in imp_top][::-1]
        vals  = [x[1] for x in imp_top][::-1]
        plt.barh(names, vals)
        plt.xlabel("Gain")
        plt.title(f"Top-{len(imp_top)} Feature Importance (Gain)")
        wandb.log({"feature_importance_gain_bar_top": wandb.Image(fig_imp)})
        plt.close(fig_imp)

    # =========================
    # (C-5.5) Top gain feature 기준 "test-case 영향" 테이블
    #   - 해석용이므로 X_test_raw2(가공 직후 raw)를 사용
    #   - prob_test / y_test 사용
    # =========================
    def build_feature_impact_tables(
        X_raw: pd.DataFrame,
        y_true: pd.Series,
        y_score: np.ndarray,
        top_features,
        threshold: float = 0.5,
        max_cat_levels: int = 15,
        high_value_quantile: float = 0.9,
    ):
        """
        반환:
          - binary_num_tbl: numeric/bool feature 영향 요약 (상위구간/flag)
          - cat_tbl: categorical feature 영향 요약 (카테고리별 top)
        """
        y_true = pd.Series(y_true).reset_index(drop=True)
        y_score = pd.Series(y_score).reset_index(drop=True)
        X_raw = X_raw.reset_index(drop=True)

        y_pred = (y_score >= threshold).astype(int)

        rows_num = []
        rows_cat = []

        for f in top_features:
            if f not in X_raw.columns:
                continue

            s = X_raw[f]

            # ---------- Categorical ----------
            if pd.api.types.is_categorical_dtype(s) or pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
                s2 = s.astype("string").fillna("__MISSING__")

                tmp = pd.DataFrame({"x": s2, "y": y_true, "pred": y_pred})
                g = tmp.groupby("x", dropna=False).agg(
                    n=("y", "size"),
                    pos=("y", "sum"),
                    pred_pos=("pred", "sum"),
                    tp=("y", lambda v: int(((tmp.loc[v.index, "y"] == 1) & (tmp.loc[v.index, "pred"] == 1)).sum())),
                )
                # recall/precision 계산
                g["recall"] = np.where(g["pos"] > 0, g["tp"] / g["pos"], np.nan)
                g["precision"] = np.where(g["pred_pos"] > 0, g["tp"] / g["pred_pos"], np.nan)

                # 너무 희귀한 레벨은 제외(표가 너무 길어짐 방지)
                g = g.sort_values(["pos", "n"], ascending=False).head(max_cat_levels).reset_index()

                for _, r in g.iterrows():
                    rows_cat.append({
                        "feature": f,
                        "level": r["x"],
                        "n_test": int(r["n"]),
                        "pos_test": int(r["pos"]),
                        "pred_pos": int(r["pred_pos"]),
                        "tp": int(r["tp"]),
                        "recall": float(r["recall"]) if pd.notna(r["recall"]) else np.nan,
                        "precision": float(r["precision"]) if pd.notna(r["precision"]) else np.nan,
                    })
                continue

            # ---------- Numeric / Bool ----------
            # bool은 1인 케이스를 바로 보자
            if pd.api.types.is_bool_dtype(s):
                mask = s.fillna(False).astype(bool)
                cond_name = "== True"
            else:
                # 숫자형이면 상위 q 구간(기본 90%)을 “high”로 정의
                # (멘토님이 말한 "피처 케이스가 몇 건"에 가장 자연스러운 정의)
                s_num = pd.to_numeric(s, errors="coerce")
                if s_num.notna().sum() == 0:
                    continue
                qv = s_num.quantile(high_value_quantile)
                mask = s_num >= qv
                cond_name = f">= q{int(high_value_quantile*100)}({qv:.4g})"

            idx = np.where(mask.values)[0]
            if len(idx) == 0:
                continue

            y_sub = y_true.iloc[idx]
            pred_sub = y_pred.iloc[idx]

            n = len(idx)
            pos = int(y_sub.sum())
            pred_pos = int(pred_sub.sum())
            tp = int(((y_sub == 1) & (pred_sub == 1)).sum())

            recall = (tp / pos) if pos > 0 else np.nan
            precision = (tp / pred_pos) if pred_pos > 0 else np.nan

            rows_num.append({
                "feature": f,
                "condition": cond_name,
                "n_test": n,
                "pos_test": pos,
                "pred_pos": pred_pos,
                "tp": tp,
                "recall": recall,
                "precision": precision,
            })

        binary_num_tbl = pd.DataFrame(rows_num).sort_values(
            ["pos_test", "n_test"], ascending=False
        ) if len(rows_num) else pd.DataFrame(
            columns=["feature","condition","n_test","pos_test","pred_pos","tp","recall","precision"]
        )

        cat_tbl = pd.DataFrame(rows_cat).sort_values(
            ["pos_test", "n_test"], ascending=False
        ) if len(rows_cat) else pd.DataFrame(
            columns=["feature","level","n_test","pos_test","pred_pos","tp","recall","precision"]
        )

        return binary_num_tbl, cat_tbl


    # ---- top gain feature list 만들기
    topN_case = int(getattr(config, "topn_case_impact", 20)) if hasattr(config, "topn_case_impact") else 20
    top_features = [name for name, _ in imp_rows[:topN_case]]

    # ===== (1) thr = 0.5 =====
    thr_05 = 0.5
    impact_num_05, impact_cat_05 = build_feature_impact_tables(
        X_raw=X_test_raw2,
        y_true=y_test,
        y_score=prob_test,
        top_features=top_features,
        threshold=thr_05,
        max_cat_levels=15,
        high_value_quantile=0.9,
    )

    print("[DEBUG] impact_num_tbl shape:", impact_num_05.shape)
    print("[DEBUG] impact_cat_tbl shape:", impact_cat_05.shape)
    print("[DEBUG] impact_num_tbl head:\n", impact_num_05.head(5))
    print("[DEBUG] impact_cat_tbl head:\n", impact_cat_05.head(5))

    wandb.log({
        "test_case_impact_numeric_topgain_thr05": wandb.Table(dataframe=impact_num_05),
        "test_case_impact_categorical_topgain_thr05": wandb.Table(dataframe=impact_cat_05),
        "case_impact_threshold_thr05": thr_05,
    })

    # ===== (2) thr = 운영 thr (chosen_thr) =====
    thr_ops = float(chosen_thr)
    impact_num_ops, impact_cat_ops = build_feature_impact_tables(
        X_raw=X_test_raw2,
        y_true=y_test,
        y_score=prob_test,
        top_features=top_features,
        threshold=thr_ops,
        max_cat_levels=15,
        high_value_quantile=0.9,
    )

    print("[DEBUG] impact_num_tbl shape:", impact_num_ops.shape)
    print("[DEBUG] impact_cat_tbl shape:", impact_cat_ops.shape)
    print("[DEBUG] impact_num_tbl head:\n", impact_num_ops.head(5))
    print("[DEBUG] impact_cat_tbl head:\n", impact_cat_ops.head(5))

    wandb.log({
        "test_case_impact_numeric_topgain_throps": wandb.Table(dataframe=impact_num_ops),
        "test_case_impact_categorical_topgain_throps": wandb.Table(dataframe=impact_cat_ops),
        "case_impact_threshold_throps": thr_ops,
    })


    # -------------------------
    # AML 패턴 분석 로깅(4가지 케이스)
    # -------------------------

    # Val 데이터 분석
    v_50_sum, v_50_det = get_pattern_analysis_tables(val_df, prob_val, 0.5)
    v_ch_sum, v_ch_det = get_pattern_analysis_tables(val_df, prob_val, chosen_thr)

    # Test 데이터 분석 (상세 분석 포함)
    t_50_sum, t_50_det = get_pattern_analysis_tables(test_df, prob_test, 0.5)
    t_ch_sum, t_ch_det = get_pattern_analysis_tables(test_df, prob_test, chosen_thr)

    # WandB 로그 전송
    wandb.log({
        "Analysis_Val/Recall_at_0.5": v_50_sum,
        "Analysis_Val/Recall_at_Chosen": v_ch_sum,
        "Analysis_Detail/Test_Detail_at_Chosen": v_50_det,
        "Analysis_Detail/Test_Detail_at_Chosen": v_ch_det,
        "Analysis_Test/Recall_at_0.5": t_50_sum,
        "Analysis_Test/Recall_at_Chosen": t_ch_sum,
        "Analysis_Detail/Test_Detail_at_Chosen": t_50_det,
        "Analysis_Detail/Test_Detail_at_Chosen": t_ch_det # 가장 중요한 데이터
    })

    # test 데이터에 대해 3000건 적발 목표 기준 패턴 분석
    t_sum_3000, t_det_3000 = get_pattern_analysis_tables_find_k(test_df, prob_test, 3000)

    wandb.log({
        "Analysis_Test/Pattern_Summary_at_Find3000": t_sum_3000,
        "Analysis_Detail/Pattern_Detail_at_Find3000": t_det_3000
    })

    # 최악의 패턴 리포트 (선택 사항)
    # t_ch_sum 데이터프레임이 비어 있지 않다면 가장 낮은 Recall 패턴을 Summary에 기록 가능

    # 1. 분석에 사용할 상위 중요 피처 추출 (Top 10개)
    # 이미 코드 상단에서 계산된 imp_rows 활용
    top_features_for_analysis = [name for name, _ in imp_rows[:10]]

    # 2. 정탐/오탐/미탐 피처 및 일자별 분석 실행 (Test 데이터 기준)
    log_detailed_error_analysis(
        split_name="test",
        df=test_df,
        y_true=y_test.values,
        y_score=prob_test,
        threshold=chosen_thr,
        top_features=top_features_for_analysis
    )

    log_detailed_error_analysis(
        split_name="test",
        df=test_df,
        y_true=y_test.values,
        y_score=prob_test,
        threshold=0.5,
        top_features=top_features_for_analysis
    )

    # 3. 패턴 중심 에러 분석 실행
    log_pattern_error_analysis(
        split_name="test",
        df=test_df,
        y_true=y_test.values,
        y_score=prob_test,
        threshold=chosen_thr
    )

    log_pattern_error_analysis(
        split_name="test",
        df=test_df,
        y_true=y_test.values,
        y_score=prob_test,
        threshold=0.5
    )

    # 2. find-3000 기준으로 정탐/오탐/미탐 분석 실행 (Test 데이터셋)
    # 이 로직은 내부적으로 workload_to_find_k_positives를 호출하여 N_required를 자동으로 계산합니다.
    log_find_k_error_analysis(
        split_name="test",
        df=test_df,
        y_true=y_test.values,
        y_score=prob_test,
        k_pos_target=3000,
        top_features=top_features_for_analysis
    )

    # =========================
    # (D) 콘솔 리포트 (test)
    # =========================
    print("=== Sweep Report ===")
    print("VAL  ROC-AUC:", val_roc_auc,  "VAL  AUPRC:", val_auprc)
    print("TEST ROC-AUC:", test_roc_auc, "TEST AUPRC:", test_auprc)
    print("Chosen thr(from VAL):", chosen_thr)
    print(classification_report(y_test, y_test_pred, zero_division=0))

    # # =========================
    # # (E) Artifacts 저장/업로드
    # # =========================
    # os.makedirs("artifacts_out", exist_ok=True)

    # preproc_path = "artifacts_out/transformer.joblib"
    # joblib.dump(transformer, preproc_path)

    # model_joblib_path = "artifacts_out/aml_model.joblib"
    # joblib.dump(xgb_model, model_joblib_path)

    # model_pkl_path = "artifacts_out/aml_model.pkl"
    # with open(model_pkl_path, "wb") as f:
    #     pickle.dump(xgb_model, f)

    # # (선택) 데이터 샘플 저장
    # data_sample_path = "artifacts_out/train_sample.csv"
    # try:
    #     sample_n = 2000
    #     X_train_raw_sample = X_train_raw.head(sample_n).copy()
    #     y_train_sample = y_train.loc[X_train_raw_sample.index].copy()
    #     X_train_raw_sample[LABEL_COL] = y_train_sample
    #     X_train_raw_sample.to_csv(data_sample_path, index=False)
    #     include_sample = True
    # except Exception:
    #     include_sample = False

    # art = wandb.Artifact(
    #     name="aml_bundle",
    #     type="model",
    #     description="Transformer(train-only fit) + XGB(scale_pos_weight sweep) + optional data sample"
    # )
    # art.add_file(preproc_path)
    # art.add_file(model_joblib_path)
    # art.add_file(model_pkl_path)
    # if include_sample:
    #     art.add_file(data_sample_path)

    # wandb.log_artifact(art)

    run.finish()

In [123]:
sweep_config = {
    "method": "bayes",
    "metric": {"name": "val_auprc", "goal": "maximize"},  # ★ 수정
    "parameters": {
        "n_estimators": {"values": [600]},
        "max_depth": {"values": [8]},
        "learning_rate": {"values": [0.1]},
        "subsample": {"values": [0.7]},
        "colsample_bytree": {"values": [1.0]},
        "reg_lambda": {"values": [5.0]},
        "reg_alpha": {"values": [0.5]},
        "min_child_weight": {"values": [10]},
        "gamma": {"values": [0.5]},
        "scale_pos_weight": {"values": [1.0]},  # ★ 추가
    }
}

# (프로젝트/엔티티 이름은 팀 기준으로 정해줘)
sweep_id = wandb.sweep(sweep_config, project="eungyulwon")

# count: 몇 번의 실험(run)을 돌릴지
wandb.agent(sweep_id, function=train_eval_sweep, count=1)

Create sweep with ID: 2zf4k5n8
Sweep URL: https://wandb.ai/0326byeol-korea-ac-kr/eungyulwon/sweeps/2zf4k5n8


wandb: Agent Starting Run: jnk2c61d with config:
wandb: 	colsample_bytree: 1
wandb: 	gamma: 0.5
wandb: 	learning_rate: 0.1
wandb: 	max_depth: 8
wandb: 	min_child_weight: 10
wandb: 	n_estimators: 600
wandb: 	reg_alpha: 0.5
wandb: 	reg_lambda: 5
wandb: 	scale_pos_weight: 1
wandb: 	subsample: 0.7
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Current files: ['.config', 'gnn_emb_val.parquet', 'drive', '.ipynb_checkpoints', 'wandb', 'gnn_emb_test.parquet', 'gnn_emb_train.parquet', 'sample_data']
=== Raw2 categorical dtypes ===
dow_cat                    category
Receiving Currency         category
Payment Currency           category
Payment Format             category
timeofday_bucket           category
Sender_Bank_Branch_ID      category
Receiver_Bank_Branch_ID    category
Sender_Entity              category
Sender_Bank_Name           category
Receiver_Entity            category
Receiver_Bank_Name         category
From Country               category
To Country                 category
From Bank                  category
To Bank                    category
dtype: object

=== Transformed X_train info ===
type(X_train): <class 'pandas.core.frame.DataFrame'>
X_train dtype: None
[DEBUG] gain_dict size: 58
[DEBUG] gain_dict keys sample: ['Amount_Received_USD', 'Amount_Paid_USD', 'IsCrossBorder', 'IsFromFATF', 'IsToFATF']
[DEBUG] i

case_impact_threshold_thr05,▁
case_impact_threshold_throps,▁
chosen_threshold,▁
desired_recall_for_thr,▁
n_train,▁
pos_test,▁
pos_train,▁
pos_val,▁
test_acc_auprc,▁
test_acc_find_1500_N_required,▁
+203,...
